# Computational Set 2: Two-Level Electronic Systems

[Open this notebook in Google Colab](https://colab.research.google.com/github/FoleyLab/chem5200-book/blob/main/notebooks/two_level_systems.ipynb)

[Jay Foley, University of North Carolina Charlotte](https://foleylab.github.io/)

#### Learning Outcomes
By the end of this workbook, students should be able to

- Explain why *any* two-level quantum system can be written in the language of spin 1/2
- Decompose an arbitrary $2\times 2$ Hermitian matrix into its identity and Pauli components, $\hat{H} = \varepsilon_0 \hat{1} + \frac{1}{2}\mathbf{d}\cdot\hat{\boldsymbol{\sigma}}$
- Relate the $\hat{\sigma}_z$ component to an **energy asymmetry** between basis states, and the $\hat{\sigma}_x$/$\hat{\sigma}_y$ components to the **coupling** that mixes them
- Predict eigenvalues, eigenvectors, and state composition using the mixing angle $\theta$ without diagonalizing
- Build and interpret the two-level Hamiltonian for three concrete chemical systems: a two-orbital LCAO dimer, a driven spin (NMR/EPR), and a donor--acceptor electron-transfer complex
- Propagate a two-level state in time and visualize the result as a Bloch vector
- Explain the physical origin of a $\hat{\sigma}_y$ contribution, and why it is absent from real, field-free electronic Hamiltonians

#### Summary

In [Computational Set 1](spin_one_half.ipynb) you built the matrix representations of $\hat{S}_x$, $\hat{S}_y$, and $\hat{S}_z$ for a spin-1/2 particle and used them to compute expectation values, find eigenstates, and verify commutation relations.

Here is the payoff. **Nothing in that formalism required the system to be a spin.** The mathematics of spin 1/2 is the mathematics of *any* quantum system with exactly two relevant states, and chemistry is full of them:

| System | The two states |
|---|---|
| A diatomic molecule in a minimal basis | atomic orbital on atom A / atom B |
| A proton in an NMR magnet | $\alpha$ / $\beta$ nuclear spin |
| A donor--acceptor complex | electron on donor / electron on acceptor |
| An ammonia molecule | umbrella pointing "up" / "down" |
| A chromophore in a laser field | ground state / first excited state |

In every one of these cases the Hamiltonian is a $2 \times 2$ Hermitian matrix, and we will see that *every* $2\times2$ Hermitian matrix is a combination of $\hat{1}$, $\hat{\sigma}_x$, $\hat{\sigma}_y$, and $\hat{\sigma}_z$. So the tools you built in Set 1 already solve all of these problems. In this notebook we make that concrete.

Two of the examples below are also the direct ancestors of models we will meet in later sets. The **driven spin** becomes the Jaynes--Cummings model when we quantize the driving field, and the **donor--acceptor complex** becomes the Holstein model when we quantize the nuclear coordinate. Everything here lives in the two-dimensional spin-1/2 space alone; the coupled spin--boson spaces come later.

---

We will use **atomic units** ($\hbar = 1$) for the abstract parts of the notebook, and switch to eV or cm$^{-1}$ where a real chemical system is involved. Each section says which.


# Import statements

Beyond `numpy`, we will need `matplotlib` for plotting, including its 3D toolkit for the Bloch sphere.

In [ ]:
import numpy as np
from numpy import linalg as la
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (registers the '3d' projection)

# atomic units
hbar = 1.0

np.set_printoptions(precision=4, suppress=True)

# Part 0: Rebuilding your toolkit

Let's start by re-creating the objects you built in Computational Set 1. We will use them constantly.

### 🚧 Your Task

Define the three Pauli matrices and the $2\times2$ identity. Recall that the spin operators are $\hat{S}_i = \frac{\hbar}{2}\hat{\sigma}_i$, so

$$
\hat\sigma_x = \begin{bmatrix} 0 & 1 \\ 1 & 0\end{bmatrix}, \qquad
\hat\sigma_y = \begin{bmatrix} 0 & -i \\ i & 0\end{bmatrix}, \qquad
\hat\sigma_z = \begin{bmatrix} 1 & 0 \\ 0 & -1\end{bmatrix}
$$

Build them as **complex** arrays (`dtype=complex`) so that adding a $\hat\sigma_y$ term later does not silently discard the imaginary part.

In [ ]:
# The 2x2 identity (given)
I2 = np.eye(2, dtype=complex)

# TODO: define the three Pauli matrices as complex numpy arrays
sigma_x = ...
sigma_y = ...
sigma_z = ...

# The z-basis kets (given). In this notebook |alpha> is "state 1" and |beta> is "state 2",
# whatever those two states physically turn out to be.
ket_alpha = np.array([[1], [0]], dtype=complex)
ket_beta = np.array([[0], [1]], dtype=complex)

bra_alpha = ket_alpha.conj().T
bra_beta = ket_beta.conj().T

print("sigma_x =\n", sigma_x)
print("sigma_y =\n", sigma_y)
print("sigma_z =\n", sigma_z)

In [ ]:
# ✅ Tests for your Pauli matrices
assert np.allclose(sigma_x @ sigma_x, I2), "sigma_x^2 should be the identity"
assert np.allclose(sigma_y @ sigma_y, I2), "sigma_y^2 should be the identity"
assert np.allclose(sigma_z @ sigma_z, I2), "sigma_z^2 should be the identity"
assert np.allclose(sigma_x @ sigma_y - sigma_y @ sigma_x, 2j * sigma_z), "[sx, sy] should equal 2i sz"
assert np.allclose(sigma_x, sigma_x.conj().T), "sigma_x should be Hermitian"
assert np.allclose(sigma_y, sigma_y.conj().T), "sigma_y should be Hermitian"
print("✅ Pauli matrices check out.")

### Two functions from Set 1, provided

You wrote `compute_bra_ket` and `compute_expectation` in Computational Set 1, so they are given to you here rather than re-derived. Read them and confirm they match what you built.

In [ ]:
def compute_bra_ket(my_bra, my_ket):
    """Compute the inner product <bra|ket> and return it as a scalar."""
    return (my_bra @ my_ket)[0, 0]


def compute_expectation(bra, Operator, ket):
    """Compute the expectation value <bra|Operator|ket> and return it as a scalar."""
    return (bra @ Operator @ ket)[0, 0]


# quick sanity checks against results you already know
assert np.isclose(compute_bra_ket(bra_alpha, ket_alpha), 1.0)
assert np.isclose(compute_bra_ket(bra_alpha, ket_beta), 0.0)
assert np.isclose(compute_expectation(bra_alpha, hbar / 2 * sigma_z, ket_alpha), 0.5)
print("✅ Toolkit from Set 1 is working.")

# Part 1: Every two-level Hamiltonian is a spin

## 🧩 The Pauli decomposition

Suppose you have a quantum system with exactly two states, $|\alpha\rangle$ and $|\beta\rangle$, which we take to be orthonormal. Any Hamiltonian for that system is a $2\times2$ **Hermitian** matrix:

$$
\hat{H} =
\begin{bmatrix}
H_{\alpha\alpha} & H_{\alpha\beta} \\
H_{\alpha\beta}^{*} & H_{\beta\beta}
\end{bmatrix}
$$

with $H_{\alpha\alpha}$ and $H_{\beta\beta}$ real. That is **four real numbers** of freedom: two diagonal elements plus the real and imaginary parts of the off-diagonal element.

The four matrices $\hat{1}, \hat\sigma_x, \hat\sigma_y, \hat\sigma_z$ also carry exactly four real degrees of freedom, and it turns out they form a complete basis for Hermitian $2\times2$ matrices. So we can always write

$$
\boxed{\;\hat{H} = \varepsilon_0\,\hat{1} + \tfrac{1}{2}\left(d_x \hat\sigma_x + d_y \hat\sigma_y + d_z \hat\sigma_z\right)
\;\equiv\; \varepsilon_0\,\hat{1} + \tfrac{1}{2}\,\mathbf{d}\cdot\hat{\boldsymbol{\sigma}}\;}
$$

with $\varepsilon_0, d_x, d_y, d_z$ all real. Writing that out,

$$
\hat{H} =
\begin{bmatrix}
\varepsilon_0 + \tfrac{d_z}{2} & \tfrac{d_x - i d_y}{2} \\[4pt]
\tfrac{d_x + i d_y}{2} & \varepsilon_0 - \tfrac{d_z}{2}
\end{bmatrix}
$$

so we can read off the physical meaning of each piece:

| Component | Physical meaning |
|---|---|
| $\varepsilon_0$ | the **average** energy of the two states -- shifts everything, changes no physics |
| $d_z$ | the **energy gap** between the two basis states, $H_{\alpha\alpha} - H_{\beta\beta}$ |
| $d_x, d_y$ | the **coupling** between the two basis states: $H_{\alpha\beta} = \tfrac{1}{2}(d_x - i d_y)$ |

That is the whole story of this notebook. **$\hat\sigma_z$ is the asymmetry; $\hat\sigma_x$ and $\hat\sigma_y$ are the coupling.** Everything else is choosing what the two states physically are.

---

### 📐 Extracting the components

Because $\mathrm{Tr}(\hat\sigma_i \hat\sigma_j) = 2\delta_{ij}$ and $\mathrm{Tr}(\hat\sigma_i) = 0$, we can pull out each component with a trace:

$$
\varepsilon_0 = \tfrac{1}{2}\mathrm{Tr}(\hat{H}), \qquad
d_i = \mathrm{Tr}\!\left(\hat\sigma_i \hat{H}\right)
$$

**Verify the second relation for yourself** before coding it: substitute the boxed expression for $\hat{H}$ into $\mathrm{Tr}(\hat\sigma_i \hat H)$ and use the two trace identities above.

### 🧪 Design Recipe: `pauli_decomposition`

Let's build a function that takes a $2\times 2$ Hermitian matrix and returns its four real Pauli components.

1. **Header** `pauli_decomposition(H)` takes a $2\times2$ complex numpy array and returns a tuple of four floats `(e0, dx, dy, dz)`.
2. **Purpose** Given in the docstring below.
3. **Examples** Two are given. Add one of your own -- a good choice is $\hat{H} = \hat{S}_x = \frac{\hbar}{2}\hat\sigma_x$, for which you should be able to predict all four numbers by inspection.
4. **Body** Complete the four lines marked `...`. Use `np.trace` and take `.real` at the end, since for a Hermitian input all four numbers are guaranteed real (any imaginary part is roundoff).
5. **Test** Run the test cell.
6. **Debug/Iterate** If a component comes out with the wrong sign, check the order of the matrix product inside the trace -- although for this particular case, does the order actually matter? (Think about the cyclic property of the trace.)

In [ ]:
def pauli_decomposition(H):
    """
    Decompose a 2x2 Hermitian matrix as H = e0 * I + (1/2)(dx*sx + dy*sy + dz*sz).

    Arguments
    ---------
    H : a 2x2 numpy array representing a Hermitian operator

    Returns
    -------
    (e0, dx, dy, dz) : a tuple of four real floats

    Examples
    --------
    pauli_decomposition(np.eye(2))            -> (1.0, 0.0, 0.0, 0.0)
    pauli_decomposition(0.5 * sigma_z)        -> (0.0, 0.0, 0.0, 1.0)
    # TODO: add one example of your own
    """
    # TODO: the average energy is half the trace
    e0 = ...

    # TODO: each d_i is the trace of sigma_i times H
    dx = ...
    dy = ...
    dz = ...

    return e0, dx, dy, dz

### 🧪 Design Recipe: `build_two_level`

Now the inverse operation: given the four components, assemble the Hamiltonian. Less scaffolding this time -- you write the whole body.

1. **Header** `build_two_level(e0, dz, dx=0.0, dy=0.0)` returns a $2\times2$ complex numpy array. Note the argument order: `dz` comes first because it is the component every example in this notebook has.
2. **Purpose** Build $\hat{H} = \varepsilon_0\hat{1} + \frac{1}{2}\mathbf{d}\cdot\hat{\boldsymbol\sigma}$.
3. **Examples** Write two in the docstring.
4. **Body** One line.
5. **Test** The test cell below checks that `build_two_level` and `pauli_decomposition` are exact inverses of one another for 500 random Hermitian matrices. That is a much stronger test than any single hand-worked example -- think about why.
6. **Debug/Iterate** A factor-of-two error here will propagate through the entire notebook, so make sure this passes.

In [ ]:
def build_two_level(e0, dz, dx=0.0, dy=0.0):
    """
    Build the 2x2 Hamiltonian H = e0 * I + (1/2)(dx*sx + dy*sy + dz*sz).

    Arguments
    ---------
    e0 : float, the average of the two diagonal energies
    dz : float, the energy difference H_aa - H_bb between the basis states
    dx : float, twice the real part of the coupling H_ab
    dy : float, minus twice the imaginary part of the coupling H_ab

    Returns
    -------
    H : a 2x2 complex numpy array

    Examples
    --------
    # TODO: write two examples of your own
    """
    # TODO: build and return the Hamiltonian
    H = ...
    return H

In [ ]:
# ✅ Test: build_two_level and pauli_decomposition must be exact inverses

# First, two cases you can check by hand
assert np.allclose(build_two_level(0.0, hbar), hbar / 2 * sigma_z), "S_z case failed"
assert np.allclose(build_two_level(0.0, 0.0, hbar), hbar / 2 * sigma_x), "S_x case failed"
assert np.allclose(pauli_decomposition(hbar / 2 * sigma_y), (0.0, 0.0, hbar, 0.0)), "S_y case failed"

# Now the round-trip test on 500 random Hermitian matrices
rng = np.random.default_rng(20250914)
for _ in range(500):
    A = rng.normal(size=(2, 2)) + 1j * rng.normal(size=(2, 2))
    H_random = A + A.conj().T                       # any A + A^dagger is Hermitian
    e0, dx, dy, dz = pauli_decomposition(H_random)
    assert np.allclose(build_two_level(e0, dz, dx, dy), H_random), "round trip failed"

print("✅ Every 2x2 Hermitian matrix really is a combination of I, sx, sy, and sz.")

### 🤔 Questions to consider

**Q1.** The round-trip test above succeeded for 500 *random* Hermitian matrices. What does that tell you that a single worked example would not?

**Q2.** Suppose you add a constant to $\varepsilon_0$. Which of the following change: the eigenvalues, the eigenvectors, the energy *differences*, the time evolution of $|\langle\beta|\psi(t)\rangle|^2$? Test your prediction numerically if you are unsure.

**Q3.** What would `pauli_decomposition` return if you handed it a non-Hermitian matrix? Would the function complain? Should it? (This is a real question about defensive programming, not a trick.)

# Part 2: The geometry of a two-level system

## 🎯 Eigenvalues without diagonalizing

Here is where the Pauli decomposition earns its keep. Because $(\mathbf{d}\cdot\hat{\boldsymbol\sigma})^2 = |\mathbf{d}|^2\,\hat{1}$ (verify this -- it follows from $\hat\sigma_i\hat\sigma_j = \delta_{ij}\hat 1 + i\epsilon_{ijk}\hat\sigma_k$), the matrix $\mathbf{d}\cdot\hat{\boldsymbol\sigma}$ squares to a multiple of the identity, so its eigenvalues can only be $\pm|\mathbf{d}|$. Therefore

$$
\boxed{\;E_{\pm} = \varepsilon_0 \pm \frac{|\mathbf{d}|}{2} = \varepsilon_0 \pm \frac{1}{2}\sqrt{d_x^2 + d_y^2 + d_z^2}\;}
$$

**No diagonalization required.** The splitting between the two levels is $|\mathbf{d}|$, full stop.

This single formula is the "avoided crossing" result, the Rabi frequency, the bonding--antibonding splitting, and the Davydov splitting, all at once. Notice what it says: because $d_z$ and $d_\perp \equiv \sqrt{d_x^2+d_y^2}$ add *in quadrature*, the splitting can never be smaller than the coupling. Even when the two basis states are exactly degenerate ($d_z = 0$), the levels are split by $d_\perp$. **Coupled states never cross.**

---

## 🧭 Eigenvectors and the mixing angle

Write $\mathbf{d}$ in spherical polar coordinates:

$$
d_z = |\mathbf{d}|\cos\theta, \qquad d_x = |\mathbf{d}|\sin\theta\cos\varphi, \qquad d_y = |\mathbf{d}|\sin\theta\sin\varphi
$$

so that

$$
\tan\theta = \frac{\sqrt{d_x^2+d_y^2}}{d_z} = \frac{d_\perp}{d_z}, \qquad \varphi = \mathrm{atan2}(d_y, d_x)
$$

Then the eigenvectors are

$$
|+\rangle = \cos\tfrac{\theta}{2}\,|\alpha\rangle + e^{i\varphi}\sin\tfrac{\theta}{2}\,|\beta\rangle, \qquad
|-\rangle = -e^{-i\varphi}\sin\tfrac{\theta}{2}\,|\alpha\rangle + \cos\tfrac{\theta}{2}\,|\beta\rangle
$$

The **mixing angle** $\theta$ is the single most useful number in two-level physics. It tells you how thoroughly the coupling has scrambled your basis states:

| Regime | $\theta$ | The eigenstates are... |
|---|---|---|
| $|d_z| \gg d_\perp$, $d_z > 0$ | $\theta \to 0$ | essentially $|\alpha\rangle$ and $|\beta\rangle$ -- **localized**, coupling is a perturbation |
| $d_z = 0$ | $\theta = \pi/2$ | exactly 50/50 mixtures -- **maximally delocalized** |
| $|d_z| \gg d_\perp$, $d_z < 0$ | $\theta \to \pi$ | localized again, but with the labels swapped |

The population of $|\alpha\rangle$ in the upper state is $\cos^2(\theta/2)$, and in the lower state $\sin^2(\theta/2)$.

> **The Bloch sphere.** The vector $\mathbf{d}$ points somewhere on a sphere, and $(\theta, \varphi)$ are just its polar angles. This is the *same* sphere on which you can plot the state itself, via $\langle\hat{\boldsymbol\sigma}\rangle$. We will use it for real in Part 4.

### 🧪 Design Recipe: `analytic_eigensystem`

Build a function that returns the eigenvalues and eigenvectors of a two-level Hamiltonian from the formulas above -- no call to `eigh`.

1. **Header** `analytic_eigensystem(e0, dz, dx=0.0, dy=0.0)` returns `(E_plus, E_minus, ket_plus, ket_minus)`, where the two kets are $2\times1$ complex column vectors.
2. **Purpose** In the docstring.
3. **Examples** Add two: what should this return for $\hat{S}_z$? For $\hat{S}_x$?
4. **Body** Complete the lines marked `...`. Use `np.hypot(dx, dy)` for $d_\perp$ and `np.arctan2` for both angles -- `arctan2` handles the quadrant correctly, which plain `arctan` does not.
5. **Test** The test cell compares your answer to `la.eigh` for many random Hamiltonians.
6. **Debug/Iterate** If eigenvalues match but eigenvectors do not, check your **half**-angles: $\theta/2$, not $\theta$.

In [ ]:
def analytic_eigensystem(e0, dz, dx=0.0, dy=0.0):
    """
    Return the eigenvalues and eigenvectors of a two-level Hamiltonian
    analytically, using the mixing angle, without diagonalizing.

    Arguments
    ---------
    e0, dz, dx, dy : floats, the Pauli components of the Hamiltonian

    Returns
    -------
    (E_plus, E_minus, ket_plus, ket_minus)
        E_plus  : float, the upper eigenvalue
        E_minus : float, the lower eigenvalue
        ket_plus, ket_minus : 2x1 complex numpy arrays

    Examples
    --------
    # TODO: add two examples of your own
    """
    # TODO: the magnitude of the perpendicular component
    d_perp = ...

    # TODO: the magnitude of the full d vector
    d_mag = ...

    # TODO: the polar (mixing) angle and the azimuthal angle
    theta = ...
    phi = ...

    # TODO: the two eigenvalues
    E_plus = ...
    E_minus = ...

    # the two eigenvectors (given, so that the phase convention is unambiguous)
    ket_plus = np.array([[np.cos(theta / 2)],
                         [np.exp(1j * phi) * np.sin(theta / 2)]], dtype=complex)
    ket_minus = np.array([[-np.exp(-1j * phi) * np.sin(theta / 2)],
                          [np.cos(theta / 2)]], dtype=complex)

    return E_plus, E_minus, ket_plus, ket_minus

In [ ]:
# ✅ Test: your analytic formulas against numpy's eigenvalue solver

rng = np.random.default_rng(1234)
for _ in range(500):
    e0, dz, dx, dy = rng.normal(size=4)
    H_test = build_two_level(e0, dz, dx, dy)
    E_plus, E_minus, kp, km = analytic_eigensystem(e0, dz, dx, dy)

    # eigenvalues agree with eigh
    assert np.allclose(np.sort(la.eigvalsh(H_test)), [E_minus, E_plus]), "eigenvalues disagree"

    # they really are eigenvectors
    assert np.allclose(H_test @ kp, E_plus * kp), "ket_plus is not an eigenvector"
    assert np.allclose(H_test @ km, E_minus * km), "ket_minus is not an eigenvector"

    # and they are orthonormal
    assert np.isclose(abs(compute_bra_ket(kp.conj().T, km)), 0.0), "eigenvectors not orthogonal"
    assert np.isclose(compute_bra_ket(kp.conj().T, kp).real, 1.0), "ket_plus not normalized"

print("✅ You can now solve any two-level problem with a pencil.")

### 🤔 Questions to consider

**Q4.** For fixed coupling $d_\perp$, sketch (by hand, before plotting anything) the two eigenvalues as a function of $d_z$ running from large and negative to large and positive. What is the minimum separation between the curves, and where does it occur?

**Q5.** Show that $\cos^2(\theta/2) + \sin^2(\theta/2) = 1$ is the statement that probability is conserved when you decompose an eigenstate in the basis $\{|\alpha\rangle, |\beta\rangle\}$.

**Q6.** In the limit $d_\perp \to 0$ with $d_z > 0$ fixed, the formulas give $\theta \to 0$, so $|+\rangle \to |\alpha\rangle$. What happens if instead $d_z \to 0$ with $d_\perp$ fixed and then you let $d_z$ pass through zero and become negative? Follow $|+\rangle$ through that process. (This continuous swap is exactly what an avoided crossing *is*.)

# Part 3: Example 1 -- the two-orbital LCAO dimer

## 🧪 The chemistry

Take the simplest possible molecular orbital problem: two atoms, one atomic orbital each. Call them $|A\rangle$ and $|B\rangle$, and map them onto our two states, $|A\rangle \equiv |\alpha\rangle$ and $|B\rangle \equiv |\beta\rangle$.

> **A note on orthogonality.** Real atomic orbitals on neighbouring atoms overlap, $\langle A|B\rangle = S \neq 0$, which strictly gives a *generalized* eigenvalue problem $\mathbf{H}\mathbf{c} = E\,\mathbf{S}\mathbf{c}$. We sidestep this by assuming the basis has already been symmetrically (Löwdin) orthogonalized, $\mathbf{S}^{-1/2}$-style, so that $\mathbf{S} = \mathbf{1}$. This is exactly the assumption Hückel theory makes. You will meet the general case in the Basis Sets set.

In this orthogonalized basis the Hamiltonian matrix elements have standard names from Hückel theory:

$$
\hat{H} = \begin{bmatrix} \alpha_A & \beta \\ \beta & \alpha_B \end{bmatrix}
$$

- $\alpha_A = \langle A|\hat{H}|A\rangle$ is the **Coulomb integral**: roughly the energy of an electron sitting in the orbital on atom A. More electronegative atom $\Rightarrow$ more negative $\alpha$.
- $\beta = \langle A|\hat{H}|B\rangle$ is the **resonance (hopping) integral**: the energy gained by an electron delocalizing between the two atoms. It is **negative** by convention, and its magnitude falls off roughly exponentially with internuclear distance.

Applying our decomposition:

$$
\hat{H} = \underbrace{\frac{\alpha_A+\alpha_B}{2}}_{\varepsilon_0}\hat{1}
\;+\; \underbrace{(\alpha_A - \alpha_B)}_{d_z}\,\frac{\hat\sigma_z}{2}
\;+\; \underbrace{2\beta}_{d_x}\,\frac{\hat\sigma_x}{2}
$$

So for the LCAO dimer,

$$
\boxed{\; d_z = \alpha_A - \alpha_B \;=\; \text{electronegativity difference}, \qquad d_x = 2\beta \;=\; \text{twice the hopping} \;}
$$

and $d_y = 0$ -- we will come back to why in Part 7.

### What the two limits mean chemically

**Homonuclear ($d_z = 0$): H$_2^+$, or the $\pi$ system of ethylene.** The mixing angle is exactly $\pi/2$, the eigenvectors are $\frac{1}{\sqrt2}(|A\rangle \pm |B\rangle)$, and the splitting is $|d_x| = 2|\beta|$. The lower state is the **bonding** MO with equal amplitude on both atoms; the upper is **antibonding**. Notice that this Hamiltonian is *literally* $\hat{S}_x$ up to a scale and shift -- so the bonding and antibonding MOs are the same vectors you found as eigenvectors of $\hat{S}_x$ in Set 1.

**Strongly heteronuclear ($|d_z| \gg |d_x|$): an ionic bond.** $\theta \to 0$, the MOs barely mix, and both electrons in the lower MO sit essentially on the more electronegative atom. This is the ionic limit.

**In between: polar covalent.** The ratio $|d_x|/|d_z| = 2|\beta|/(\alpha_A-\alpha_B)$ is the single knob controlling how covalent the bond is.

> 🧠 **Mixed-valence chemistry.** This same ratio is the basis of the **Robin--Day classification** of mixed-valence complexes. Class I ($|\beta| \ll |\Delta|$) means fully trapped valences; Class III ($|\beta| \gg |\Delta|$) means fully delocalized -- the Creutz--Taube ion $[(\mathrm{NH_3})_5\mathrm{Ru}$-pyrazine-$\mathrm{Ru(NH_3)_5}]^{5+}$ is the textbook Class III case. Class II is the interesting middle.

### 🚧 Your Task: build and decompose a heteronuclear dimer

Use $\alpha_A = -11.4$ eV and $\alpha_B = -13.6$ eV (atom B is the more electronegative one) with $\beta = -2.7$ eV. All energies in eV for this section.

Build the Hamiltonian, decompose it, and confirm the components are what the boxed formulas predict.

In [ ]:
# Hueckel parameters, in eV
alpha_A = -11.4
alpha_B = -13.6
beta = -2.7

# TODO: build the 2x2 Hueckel Hamiltonian. You may use np.array directly,
#       or build_two_level -- try it both ways and check they agree!
H_dimer = ...

# TODO: decompose it
e0_dimer, dx_dimer, dy_dimer, dz_dimer = ...

print(f"e0 = {e0_dimer:8.3f} eV   (predicted (alpha_A+alpha_B)/2 = {(alpha_A+alpha_B)/2:.3f})")
print(f"dz = {dz_dimer:8.3f} eV   (predicted alpha_A - alpha_B    = {alpha_A-alpha_B:.3f})")
print(f"dx = {dx_dimer:8.3f} eV   (predicted 2*beta               = {2*beta:.3f})")
print(f"dy = {dy_dimer:8.3f} eV   (predicted 0)")

In [ ]:
# ✅ Tests
assert np.isclose(e0_dimer, (alpha_A + alpha_B) / 2)
assert np.isclose(dz_dimer, alpha_A - alpha_B)
assert np.isclose(dx_dimer, 2 * beta)
assert np.isclose(dy_dimer, 0.0)
print("✅ The Hueckel dimer is a spin-1/2 problem in disguise.")

### 🚧 Your Task: the molecular orbitals

Use `analytic_eigensystem` to get the bonding and antibonding MOs, then report:

- the two MO energies and the HOMO--LUMO gap
- the mixing angle $\theta$ in degrees
- the fraction of the **bonding** MO residing on each atom, $|c_A|^2$ and $|c_B|^2$

Before you run it: which atom should carry more of the bonding MO, and why?

In [ ]:
# TODO: get the eigensystem
E_anti, E_bond, ket_anti, ket_bond = ...

# TODO: the mixing angle, in degrees
theta_dimer = ...

# TODO: populations on each atom in the BONDING MO
pop_A = ...
pop_B = ...

print(f"antibonding MO energy = {E_anti:8.3f} eV")
print(f"bonding     MO energy = {E_bond:8.3f} eV")
print(f"splitting             = {E_anti - E_bond:8.3f} eV   (= |d|)")
print(f"mixing angle theta    = {theta_dimer:8.2f} degrees")
print()
print(f"bonding MO:  |c_A|^2 = {pop_A:.3f},  |c_B|^2 = {pop_B:.3f}")

In [ ]:
# ✅ Tests
assert np.isclose(E_anti - E_bond, np.sqrt(dz_dimer**2 + dx_dimer**2)), "splitting should equal |d|"
assert np.isclose(pop_A + pop_B, 1.0), "populations must sum to 1"
assert pop_B > pop_A, "more of the bonding MO should sit on the more electronegative atom"
# cross-check against numpy
vals_check, vecs_check = la.eigh(H_dimer)
assert np.allclose(np.sort([E_bond, E_anti]), vals_check)
print("✅ MO analysis confirmed against la.eigh.")

### 📊 The avoided crossing

Now the picture that makes the whole notebook click. Hold the coupling $\beta$ fixed and sweep the electronegativity difference $d_z = \alpha_A - \alpha_B$ from strongly negative to strongly positive. Plot:

**(left)** the two MO energies versus $d_z$, together with the *uncoupled* diagonal energies $\varepsilon_0 \pm d_z/2$ (the dashed "diabatic" lines that would cross at $d_z=0$);

**(right)** the composition of the lower MO, $|c_A|^2 = \sin^2(\theta/2)$ and $|c_B|^2 = \cos^2(\theta/2)$, versus $d_z$.

Most of the plotting is written for you. You supply the physics.

In [ ]:
dz_scan = np.linspace(-10.0, 10.0, 401)
beta_fixed = -1.0                      # eV, held constant across the scan
e0_fixed = 0.0                         # set the average energy to zero for a clean plot

E_upper = np.zeros_like(dz_scan)
E_lower = np.zeros_like(dz_scan)
frac_A_lower = np.zeros_like(dz_scan)

for i, dz_i in enumerate(dz_scan):
    # TODO: for this value of dz, get the eigenvalues and the lower eigenvector
    #       (remember: d_x = 2*beta)
    Ep, Em, kp, km = ...

    # TODO: store the upper and lower energies
    E_upper[i] = ...
    E_lower[i] = ...

    # TODO: store the fraction of the LOWER state on |A> (i.e. on |alpha>)
    frac_A_lower[i] = ...

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.2))

# --- left panel: the avoided crossing ---
ax1.plot(dz_scan, e0_fixed + dz_scan / 2, "k--", lw=1, label=r"uncoupled $\alpha_A$")
ax1.plot(dz_scan, e0_fixed - dz_scan / 2, "k:", lw=1, label=r"uncoupled $\alpha_B$")
ax1.plot(dz_scan, E_upper, color="crimson", lw=2, label="antibonding MO")
ax1.plot(dz_scan, E_lower, color="royalblue", lw=2, label="bonding MO")
ax1.annotate("", xy=(0, E_upper[len(dz_scan) // 2]), xytext=(0, E_lower[len(dz_scan) // 2]),
             arrowprops=dict(arrowstyle="<->", color="darkgreen", lw=1.5))
ax1.text(0.7, -0.35, r"$2|\beta| = |d_x|$", color="darkgreen", fontsize=11)
ax1.set_xlabel(r"$d_z = \alpha_A - \alpha_B$  (eV)")
ax1.set_ylabel("energy (eV)")
ax1.set_title("Avoided crossing in an LCAO dimer")
ax1.legend(fontsize=9)

# --- right panel: composition of the lower MO ---
ax2.plot(dz_scan, frac_A_lower, color="royalblue", lw=2, label=r"$|c_A|^2$")
ax2.plot(dz_scan, 1 - frac_A_lower, color="crimson", lw=2, label=r"$|c_B|^2$")
ax2.axhline(0.5, color="gray", ls=":", lw=1)
ax2.set_xlabel(r"$d_z = \alpha_A - \alpha_B$  (eV)")
ax2.set_ylabel("population in the bonding MO")
ax2.set_title("The bond goes ionic-covalent-ionic")
ax2.set_ylim(-0.03, 1.03)
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# ✅ Tests on the scan
mid = len(dz_scan) // 2
assert np.isclose(dz_scan[mid], 0.0), "the scan should pass through dz = 0"
assert np.isclose(E_upper[mid] - E_lower[mid], 2 * abs(beta_fixed)), \
    "at dz = 0 the gap should be exactly 2|beta|"
assert np.isclose(frac_A_lower[mid], 0.5), "at dz = 0 the bonding MO should be 50/50"
assert np.all(E_upper - E_lower >= 2 * abs(beta_fixed) - 1e-9), \
    "the gap can never fall below 2|beta| -- coupled levels do not cross"
assert frac_A_lower[0] > 0.9 and frac_A_lower[-1] < 0.1, \
    "at large |dz| the bonding MO should localize on the lower-energy atom"
print("✅ Avoided crossing verified: the gap never closes.")

### 🤔 Questions to consider

**Q7.** In the left panel, the dashed lines cross at $d_z = 0$ but the solid lines do not. In one sentence, what physically prevents the two MOs from becoming degenerate?

**Q8.** The right panel is a smooth switch, not a sharp one. Over what range of $d_z$ (in units of $|\beta|$) does the bonding MO go from 90% on A to 10% on A? What does that tell you about how large an electronegativity difference is needed to make a bond "ionic"?

**Q9.** For the homonuclear case, we said the Hamiltonian is $\hat{S}_x$ up to a scale and shift. Look back at your eigenvectors of $\hat{S}_x$ from Computational Set 1. Which one is the bonding MO, and which is antibonding? Careful -- $\beta$ is negative.

**Q10.** Suppose you stretch the bond. Which of $\alpha_A, \alpha_B, \beta$ changes most, and in which direction? Sketch what happens to the left panel as you pull the atoms apart. (This is the beginning of a dissociation curve -- and the beginning of why restricted Hartree--Fock fails at dissociation, which you will see later in the course.)

# Part 4: Dynamics -- the Bloch vector

So far everything has been static structure. Now let's make the state move.

## 🌀 The state as a vector on a sphere

For a normalized two-level state $|\psi\rangle$, define the **Bloch vector**

$$
\mathbf{s} = \langle \hat{\boldsymbol\sigma}\rangle =
\big(\langle\psi|\hat\sigma_x|\psi\rangle,\; \langle\psi|\hat\sigma_y|\psi\rangle,\; \langle\psi|\hat\sigma_z|\psi\rangle\big)
$$

Three real numbers. For any *pure* state, $|\mathbf{s}| = 1$ exactly, so the state lives on the surface of a unit sphere -- the **Bloch sphere**. The north pole is $|\alpha\rangle$, the south pole is $|\beta\rangle$, and every superposition is somewhere in between. The equator is where the two states are equally weighted, and the *longitude* on the equator is the relative phase.

## ⚡ The equation of motion

Apply the Heisenberg equation to each Pauli matrix, using $[\hat\sigma_a, \hat\sigma_b] = 2i\,\epsilon_{abc}\hat\sigma_c$ (which you verified for one case in Set 1):

$$
\frac{d\langle\hat\sigma_a\rangle}{dt} = \frac{i}{\hbar}\big\langle[\hat{H}, \hat\sigma_a]\big\rangle
= \frac{i}{2\hbar} d_b \big\langle [\hat\sigma_b, \hat\sigma_a]\big\rangle
$$

Working through the Levi-Civita algebra gives a remarkably clean result:

$$
\boxed{\;\frac{d\mathbf{s}}{dt} = \frac{1}{\hbar}\,\mathbf{d}\times\mathbf{s}\;}
$$

**The Bloch vector precesses about $\mathbf{d}$ at angular frequency $|\mathbf{d}|/\hbar$.** That is the entire dynamics of every two-level system. It is exactly the equation for a classical gyroscope in a torque field, or a magnetic moment in a magnetic field -- which is not a coincidence, since for a real spin in a real field $\mathbf{d} = -\gamma\hbar\mathbf{B}$.

Two immediate consequences worth internalizing:

1. **The component of $\mathbf{s}$ along $\mathbf{d}$ is conserved.** A cross product is perpendicular to both its arguments, so $\mathbf{d}\cdot\mathbf{s}$ never changes. This is just energy conservation, since $\langle\hat{H}\rangle = \varepsilon_0 + \tfrac{1}{2}\mathbf{d}\cdot\mathbf{s}$.
2. **The precession frequency is the level splitting divided by $\hbar$.** $|\mathbf{d}| = E_+ - E_-$, so $\omega_{\rm prec} = (E_+-E_-)/\hbar$ -- the Bohr frequency. You have already met this idea in the postulates chapter; here you can watch it.

Notice also that $\varepsilon_0$ is nowhere in the equation of motion. An overall energy shift produces an overall phase, which no measurement on this system can see. (Compare your answer to **Q2**.)

### 🧪 Design Recipe: `bloch_vector`

1. **Header** `bloch_vector(psi)` takes a $2\times1$ complex column vector and returns a length-3 real numpy array.
2. **Purpose** In the docstring.
3. **Examples** Add two of your own. Good choices: what is the Bloch vector of $|\alpha\rangle$? Of $\frac{1}{\sqrt2}(|\alpha\rangle+|\beta\rangle)$?
4. **Body** Reuse `compute_expectation`. Take `.real` -- expectation values of Hermitian operators are real by construction.
5. **Test** Below.
6. **Debug/Iterate** If $|\mathbf{s}| \neq 1$, check that your input state is normalized.

In [ ]:
def bloch_vector(psi):
    """
    Compute the Bloch vector (<sx>, <sy>, <sz>) of a normalized two-level state.

    Arguments
    ---------
    psi : a 2x1 complex numpy array (a normalized ket)

    Returns
    -------
    s : a length-3 real numpy array

    Examples
    --------
    # TODO: add two examples of your own
    """
    psi = psi.reshape(2, 1)
    bra = psi.conj().T

    # TODO: compute the three expectation values and pack them into an array
    s = ...
    return s

In [ ]:
# ✅ Tests for bloch_vector
assert np.allclose(bloch_vector(ket_alpha), [0, 0, 1]), "|alpha> should sit at the north pole"
assert np.allclose(bloch_vector(ket_beta), [0, 0, -1]), "|beta> should sit at the south pole"
assert np.allclose(bloch_vector((ket_alpha + ket_beta) / np.sqrt(2)), [1, 0, 0]), "should be +x"
assert np.allclose(bloch_vector((ket_alpha + 1j * ket_beta) / np.sqrt(2)), [0, 1, 0]), "should be +y"

# every pure state must land on the unit sphere
rng = np.random.default_rng(7)
for _ in range(200):
    v = rng.normal(size=(2, 1)) + 1j * rng.normal(size=(2, 1))
    v = v / la.norm(v)
    assert np.isclose(la.norm(bloch_vector(v)), 1.0), "pure states must have |s| = 1"
print("✅ bloch_vector works, and pure states live on the sphere.")

### 🧪 Design Recipe: `propagate` (guided template)

To watch the precession we need to solve the time-dependent Schrödinger equation. For a **time-independent** $\hat{H}$ we do not need any numerical integrator at all -- we can write the exact answer using the eigenvectors, which you already know how to get.

**The idea.** Expand the initial state in the eigenbasis of $\hat{H}$:

$$
|\psi(0)\rangle = \sum_n c_n |n\rangle, \qquad c_n = \langle n|\psi(0)\rangle
$$

Each eigenstate simply picks up a phase, so

$$
|\psi(t)\rangle = \sum_n c_n\, e^{-i E_n t/\hbar}\,|n\rangle
$$

That is the *only* thing time evolution does in an eigenbasis. In matrix form, with `vals, vecs = la.eigh(H)`:

- `c = vecs.conj().T @ psi0` projects onto the eigenbasis (each row of `vecs.conj().T` is a bra $\langle n|$)
- multiply each $c_n$ by $e^{-iE_n t/\hbar}$
- `vecs @ (...)` transforms back to the original basis

1. **Header** `propagate(H, psi0, times)` returns an array of shape `(len(times), 2)` -- one row per time, each row a state.
2. **Purpose** In the docstring.
3. **Examples** Predict: if `psi0` is an eigenstate of `H`, what should `bloch_vector` of every row be?
4. **Body** Three lines marked `...`.
5. **Test** Below -- norm conservation and the conservation law from point 1 above.
6. **Debug/Iterate** If the state grows or shrinks, you have a sign or conjugation error. If it evolves backwards, check the sign in the exponent.

In [ ]:
def propagate(H, psi0, times):
    """
    Exactly propagate a two-level state under a time-INDEPENDENT Hamiltonian.

    Arguments
    ---------
    H     : 2x2 Hermitian numpy array
    psi0  : 2x1 complex numpy array, the initial state
    times : 1D array of times at which to report the state

    Returns
    -------
    psis : complex array of shape (len(times), 2); psis[i] is the state at times[i]

    Examples
    --------
    # TODO: what do you expect if psi0 is an eigenstate of H?
    """
    times = np.atleast_1d(times)

    # TODO: diagonalize H
    vals, vecs = ...

    # TODO: project the initial state onto the eigenbasis -> a 2x1 array of coefficients
    c = ...

    psis = np.zeros((len(times), 2), dtype=complex)
    for i, t in enumerate(times):
        # TODO: phase each coefficient, then transform back to the original basis
        #       hint: np.exp(-1j * vals[:, None] * t / hbar) * c   phases them all at once
        psis[i] = ...

    return psis

In [ ]:
# ✅ Tests for propagate

H_demo = build_two_level(0.3, 1.1, -0.7, 0.4)      # a generic tilted Hamiltonian
d_demo = np.array([-0.7, 0.4, 1.1])                 # its d vector, (dx, dy, dz)
t_demo = np.linspace(0, 12, 1201)
psis_demo = propagate(H_demo, ket_alpha, t_demo)
s_demo = np.array([bloch_vector(p) for p in psis_demo])

# 1. the norm is conserved
assert np.allclose(la.norm(psis_demo, axis=1), 1.0), "propagation must preserve the norm"

# 2. the Bloch vector stays on the unit sphere
assert np.allclose(la.norm(s_demo, axis=1), 1.0), "the state must stay pure"

# 3. the projection along d is conserved (energy conservation)
proj = s_demo @ d_demo / la.norm(d_demo)
assert np.ptp(proj) < 1e-10, "the component of s along d must not change"

# 4. an eigenstate does not move at all
_, _, kp_demo, _ = analytic_eigensystem(0.3, 1.1, -0.7, 0.4)
s_eig = np.array([bloch_vector(p) for p in propagate(H_demo, kp_demo, t_demo)])
assert np.allclose(s_eig, s_eig[0]), "an eigenstate should be stationary"

# 5. the precession period matches the level splitting
period_expected = 2 * np.pi * hbar / la.norm(d_demo)
assert np.allclose(bloch_vector(propagate(H_demo, ket_alpha, [period_expected])[0]),
                   bloch_vector(ket_alpha), atol=1e-8), "one period should return s to its start"

print(f"✅ propagate works. Precession period = {period_expected:.4f} a.u. = 2*pi*hbar/|d|")

### 📊 Visualizing the precession

Let's watch it. The cell below draws the unit Bloch sphere, the axis $\hat{\mathbf{d}}$, and the trajectory traced by $\mathbf{s}(t)$ starting from $|\alpha\rangle$.

You supply the Hamiltonian and the trajectory; the sphere-drawing is provided.

In [ ]:
# TODO: choose Pauli components for a "tilted" Hamiltonian, propagate |alpha>,
#       and collect the Bloch vectors.
#       Try dz = 1.0, dx = 1.4, dy = 0.0 first, then experiment.
e0_b, dz_b, dx_b, dy_b = ..., ..., ..., ...

H_bloch = ...
# one full precession period is 2*pi*hbar/|d|
t_bloch = np.linspace(0, 2 * np.pi * hbar / np.sqrt(dx_b**2 + dy_b**2 + dz_b**2), 400)
psis_bloch = ...
s_traj = ...   # shape should be (400, 3)

# ---------------- provided plotting ----------------
d_hat = np.array([dx_b, dy_b, dz_b]) / np.sqrt(dx_b**2 + dy_b**2 + dz_b**2)

fig = plt.figure(figsize=(6.5, 6))
ax = fig.add_subplot(111, projection="3d")

# wireframe sphere
u = np.linspace(0, 2 * np.pi, 60)
v = np.linspace(0, np.pi, 31)
ax.plot_wireframe(np.outer(np.cos(u), np.sin(v)),
                  np.outer(np.sin(u), np.sin(v)),
                  np.outer(np.ones_like(u), np.cos(v)),
                  color="lightgray", lw=0.35, rstride=3, cstride=4, alpha=0.7)

# coordinate axes, drawn all the way through the sphere
for vec, lab in [((1.45, 0, 0), "x"), ((0, 1.45, 0), "y"), ((0, 0, 1.35), "z")]:
    v_arr = np.array(vec)
    ax.plot(*zip(-0.9 * v_arr, v_arr), color="gray", lw=0.7)
    ax.text(*v_arr, lab, fontsize=11, color="gray")

# the precession axis d-hat
ax.quiver(0, 0, 0, *(1.15 * d_hat), color="darkgreen", lw=2.5, arrow_length_ratio=0.12)
ax.text(*(1.30 * d_hat), r"$\hat{\mathbf{d}}$", color="darkgreen", fontsize=14)

# the trajectory
ax.plot(s_traj[:, 0], s_traj[:, 1], s_traj[:, 2], color="crimson", lw=2.2)
ax.scatter(*s_traj[0], color="black", s=45, zorder=6)
ax.text(-0.05, 0.30, 1.14, r"$\mathbf{s}(0) = |\alpha\rangle$", fontsize=11)
ax.scatter(0, 0, -1, color="black", s=25, zorder=6)
ax.text(-0.05, 0.28, -1.22, r"$|\beta\rangle$", fontsize=11)

# a quarter of the way around, to show which way it goes
ax.scatter(*s_traj[len(s_traj) // 4], color="crimson", s=45, zorder=6)

ax.set_box_aspect((1, 1, 1))
ax.set_xlim(-1, 1); ax.set_ylim(-1, 1); ax.set_zlim(-1, 1)
ax.set_xticks([-1, 0, 1]); ax.set_yticks([-1, 0, 1]); ax.set_zticks([-1, 0, 1])
ax.set_title(r"Bloch vector precessing about $\mathbf{d}$"
             "\n(black = start, red dot = one quarter period later)", fontsize=11, pad=0)
ax.view_init(elev=20, azim=38)
plt.tight_layout()
plt.show()

print(f"cone half-angle between s(0) and d-hat: {np.degrees(np.arccos(s_traj[0] @ d_hat)):.1f} degrees")
print(f"max deviation of s.d-hat over the trajectory: {np.ptp(s_traj @ d_hat):.2e}  (should be ~0)")

### 🤔 Questions to consider

**Q11.** The trajectory is a circle, not a great circle. What sets the radius of that circle? Predict what happens to it as you make $d_x$ smaller and smaller relative to $d_z$, then change the numbers and check.

**Q12.** How would you choose $\mathbf{d}$ so that the state starting at $|\alpha\rangle$ reaches $|\beta\rangle$ exactly? (Hint: the trajectory must be a great circle through both poles.) This is the question that Part 5 answers physically.

**Q13.** The state returns to where it started after one precession period. Does the *ket* also return exactly to $|\psi(0)\rangle$, or only to something physically equivalent? Test it: compare `psis_bloch[-1]` to `ket_alpha` numerically, and explain any discrepancy.

# Part 5: Example 2 -- a driven two-level system (NMR, EPR, and lasers)

## 🧲 The physical setup

A spin-1/2 nucleus in a strong static field $B_0\hat{z}$ has the Zeeman Hamiltonian

$$
\hat{H}_0 = -\gamma B_0 \hat{S}_z = \frac{\hbar\omega_0}{2}\hat\sigma_z, \qquad \omega_0 = -\gamma B_0
$$

Pure $\hat\sigma_z$: the two spin states are split by $\hbar\omega_0$, the **Larmor frequency**. For a proton at 11.7 T this is 500 MHz -- the number on the front of the NMR spectrometer. **Nothing happens** in this Hamiltonian: $|\alpha\rangle$ and $|\beta\rangle$ are eigenstates and the Bloch vector just spins about the $z$ axis.

To make something happen we add a weak field rotating in the $xy$ plane at frequency $\omega$ (in NMR this is the RF coil; for an electronic transition driven by a laser it is the optical field in the dipole approximation):

$$
\hat{H}(t) = \frac{\hbar\omega_0}{2}\hat\sigma_z
+ \frac{\hbar\omega_1}{2}\Big[\cos(\omega t + \varphi)\,\hat\sigma_x + \sin(\omega t + \varphi)\,\hat\sigma_y\Big],
\qquad \omega_1 = -\gamma B_1
$$

This is genuinely time dependent, so `propagate` does not apply. But there is a classic trick.

## 🔄 The rotating frame

Move into a frame rotating about $z$ at the drive frequency $\omega$, via $|\tilde\psi\rangle = e^{+i\omega t \hat\sigma_z/2}|\psi\rangle$. In that frame the drive stops moving, and the transformed Hamiltonian is **time independent**:

$$
\boxed{\;\hat{H}_{\rm rot} = \frac{\hbar}{2}\Big[\underbrace{(\omega_0 - \omega)}_{\text{detuning }\Delta}\hat\sigma_z
+ \omega_1\big(\cos\varphi\,\hat\sigma_x + \sin\varphi\,\hat\sigma_y\big)\Big]\;}
$$

In our language, $d_z = \hbar\Delta$, $d_x = \hbar\omega_1\cos\varphi$, $d_y = \hbar\omega_1\sin\varphi$.

Three things to notice, because each is a piece of real spectroscopic practice:

- **The detuning is the $\sigma_z$ knob.** On resonance ($\omega = \omega_0$) it vanishes entirely and $\mathbf{d}$ lies flat in the $xy$ plane. Off resonance, $\mathbf{d}$ tilts toward the poles.
- **The pulse phase $\varphi$ is the $\sigma_y$ knob.** Setting $\varphi=0$ gives a pure $\hat\sigma_x$ term (an "$x$-pulse"), $\varphi=\pi/2$ gives a pure $\hat\sigma_y$ term (a "$y$-pulse"). This is not jargon -- it is literally the phase of the RF pulse, and it is why every pulse sequence diagram in an NMR textbook labels its pulses $x$, $y$, $-x$, $-y$.
- **The tilted-field geometry of Part 4 is now physical.** $\mathbf{d}$ has a $z$ component set by how far off resonance you are, and a transverse component set by how hard you drive.

## 🎯 Rabi oscillations

Apply the boxed eigenvalue formula. The generalized Rabi frequency is

$$
\Omega = \frac{|\mathbf{d}|}{\hbar} = \sqrt{\Delta^2 + \omega_1^2}
$$

and starting from $|\alpha\rangle$ the population transferred is

$$
\boxed{\;P_\beta(t) = \frac{\omega_1^2}{\Omega^2}\sin^2\!\left(\frac{\Omega t}{2}\right)\;}
$$

Read that carefully. The **rate** of oscillation always increases with detuning, but the **amplitude** falls off as $\omega_1^2/(\Delta^2+\omega_1^2)$ -- a Lorentzian of width $\omega_1$. Only exactly on resonance can you achieve complete population inversion. This is the whole physics of a spectroscopic lineshape and of pulse selectivity, in one formula.

### 🚧 Your Task: Rabi oscillations at several detunings

Work in units where $\omega_1 = 1$, so time is measured in units of $1/\omega_1$. Build $\hat{H}_{\rm rot}$ for $\varphi = 0$ and a range of detunings, propagate $|\alpha\rangle$, and plot $P_\beta(t)$.

Then compare your numerical curves to the boxed analytic formula.

In [ ]:
omega_1 = 1.0                       # drive strength (sets our unit of time)
phi_pulse = 0.0                     # pulse phase: 0 is an "x-pulse"
detunings = [0.0, 1.0, 2.0, 4.0]    # in units of omega_1
t_rabi = np.linspace(0, 4 * np.pi, 800)

plt.figure(figsize=(7.5, 4.5))
for Delta in detunings:
    # TODO: build the rotating-frame Hamiltonian for this detuning
    #       remember d_z = hbar*Delta, d_x = hbar*omega_1*cos(phi), d_y = hbar*omega_1*sin(phi)
    H_rot = ...

    # TODO: propagate |alpha> and extract P_beta(t) = |<beta|psi(t)>|^2
    psis_rabi = ...
    P_beta = ...

    # TODO: the analytic prediction
    Omega = ...
    P_beta_analytic = ...

    line, = plt.plot(t_rabi, P_beta, lw=2, label=rf"$\Delta/\omega_1 = {Delta:.0f}$")
    plt.plot(t_rabi, P_beta_analytic, "k--", lw=1, alpha=0.7)

plt.xlabel(r"$\omega_1 t$")
plt.ylabel(r"$P_\beta(t)$")
plt.title("Rabi oscillations (dashed black = analytic formula)")
plt.ylim(-0.03, 1.05)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ✅ Tests: numerics must reproduce the analytic Rabi formula

for Delta in [0.0, 0.5, 2.0, -1.3]:
    H_chk = build_two_level(0.0, hbar * Delta, hbar * omega_1, 0.0)
    P_num = np.abs(propagate(H_chk, ket_alpha, t_rabi)[:, 1]) ** 2
    Om = np.sqrt(Delta**2 + omega_1**2)
    P_ana = (omega_1**2 / Om**2) * np.sin(Om * t_rabi / 2) ** 2
    assert np.allclose(P_num, P_ana, atol=1e-9), f"Rabi formula failed at Delta = {Delta}"

# maximum transfer really is the Lorentzian omega_1^2 / (Delta^2 + omega_1^2)
for Delta in [0.0, 1.0, 3.0]:
    H_chk = build_two_level(0.0, hbar * Delta, hbar * omega_1, 0.0)
    P_max = np.max(np.abs(propagate(H_chk, ket_alpha, np.linspace(0, 30, 6000))[:, 1]) ** 2)
    assert np.isclose(P_max, omega_1**2 / (Delta**2 + omega_1**2), atol=1e-4)

print("✅ Rabi oscillations verified against P = (w1^2/Omega^2) sin^2(Omega t / 2).")

### 🚧 Your Task: $\pi$ and $\pi/2$ pulses, and what the phase does

On resonance ($\Delta = 0$), the Bloch vector precesses about an axis lying in the $xy$ plane, at angle $\varphi$ from the $x$ axis. Starting at the north pole, the state sweeps down a great circle -- so with the right duration you can put it anywhere you like:

- a **$\pi$ pulse**, $t = \pi/\omega_1$, drives $|\alpha\rangle \to |\beta\rangle$ completely (population inversion)
- a **$\pi/2$ pulse**, $t = \pi/(2\omega_1)$, creates the equal superposition that lands on the equator -- the coherence that actually generates the free-induction-decay signal in an NMR experiment

Verify both, and then show that the pulse *phase* $\varphi$ chooses **where on the equator** a $\pi/2$ pulse lands you.

In [ ]:
# --- pi pulse on resonance ---
H_res = build_two_level(0.0, 0.0, hbar * omega_1, 0.0)
t_pi = np.pi / omega_1

# TODO: propagate for exactly t_pi and read off the population in |beta>
psi_after_pi = ...
P_beta_pi = ...

print(f"pi pulse:    P_beta = {P_beta_pi:.10f}   (complete inversion)")
print(f"             Bloch vector = {np.round(bloch_vector(psi_after_pi), 6)}")
print()

# --- pi/2 pulses with two different phases ---
t_pi2 = np.pi / (2 * omega_1)
for phi, name in [(0.0, "x-pulse (phi = 0)   "), (np.pi / 2, "y-pulse (phi = pi/2)")]:
    # TODO: build the on-resonance Hamiltonian with this pulse phase
    H_phi = ...
    # TODO: propagate for t_pi2 and get the Bloch vector
    s_after = ...
    print(f"pi/2 {name}: Bloch vector = {np.round(s_after, 6)}")

In [ ]:
# ✅ Tests for the pulses
assert np.isclose(P_beta_pi, 1.0, atol=1e-12), "a resonant pi pulse must fully invert the population"

s_x_pulse = bloch_vector(propagate(build_two_level(0, 0, hbar * omega_1, 0.0),
                                   ket_alpha, [t_pi2])[0])
s_y_pulse = bloch_vector(propagate(build_two_level(0, 0, 0.0, hbar * omega_1),
                                   ket_alpha, [t_pi2])[0])

assert np.allclose(s_x_pulse, [0, -1, 0], atol=1e-10), "an x-pulse should rotate |alpha> onto -y"
assert np.allclose(s_y_pulse, [1, 0, 0], atol=1e-10), "a y-pulse should rotate |alpha> onto +x"
assert np.isclose(s_x_pulse @ s_y_pulse, 0.0, atol=1e-10), \
    "the two pulses leave the state 90 degrees apart on the equator"

print("✅ pi and pi/2 pulses confirmed; the pulse phase selects the rotation axis.")

### 🤔 Questions to consider

**Q14.** A $\pi$ pulse at $\Delta = 0$ gives complete inversion. Using the amplitude factor $\omega_1^2/\Omega^2$, how far off resonance can you be before a "$\pi$ pulse" transfers less than half the population? Express your answer in units of $\omega_1$. This is the origin of **pulse selectivity**: a weak, long pulse excites a narrow band of the spectrum; a strong, short pulse excites everything.

**Q15.** An $x$-pulse takes $|\alpha\rangle$ to $-\hat{y}$ and a $y$-pulse takes it to $+\hat{x}$. Convince yourself with the right-hand rule that this is what $\frac{d\mathbf{s}}{dt} = \frac{1}{\hbar}\mathbf{d}\times\mathbf{s}$ predicts.

**Q16.** In the rotating frame the Hamiltonian is time independent, which is why `propagate` worked. What did we give up? (Consider: what does the physical state look like in the lab frame, and at what frequency is it moving?)

### 🔮 Looking ahead: this is the Jaynes--Cummings model waiting to happen

Write the transverse part of $\hat{H}_{\rm rot}$ using the raising and lowering operators $\hat\sigma_\pm = \frac{1}{2}(\hat\sigma_x \pm i\hat\sigma_y)$, which you will meet properly next time:

$$
\frac{\hbar\omega_1}{2}\big(\cos\varphi\,\hat\sigma_x + \sin\varphi\,\hat\sigma_y\big)
= \frac{\hbar\omega_1}{2}\Big(e^{-i\varphi}\hat\sigma_+ + e^{+i\varphi}\hat\sigma_-\Big)
$$

The drive **raises** the two-level system while supplying a phase, or **lowers** it while absorbing one. Here the field is a classical number, $\omega_1 \propto B_1$, and it is inexhaustible -- the atom can cycle forever and the field never notices.

Now promote the field to a quantum degree of freedom. The phase factors become photon annihilation and creation operators, $e^{-i\varphi} \to \hat{a}$ and $e^{+i\varphi} \to \hat{a}^\dagger$, and the Hamiltonian becomes

$$
\hat{H}_{\rm JC} = \frac{\hbar\omega_0}{2}\hat\sigma_z + \hbar\omega\,\hat{a}^\dagger\hat{a}
+ \hbar g\big(\hat{a}\,\hat\sigma_+ + \hat{a}^\dagger\hat\sigma_-\big)
$$

which is exactly the Tavis--Cummings Hamiltonian you will see in [Computational Set 4](two_qubit_entangled_DR.ipynb), for a single emitter. The Hilbert space is no longer two-dimensional -- it is spin $\otimes$ boson -- and $\omega_1$ is replaced by $2g\sqrt{n+1}$, so even the **vacuum** ($n=0$) drives Rabi oscillations. That is vacuum Rabi splitting, and it is the whole basis of polaritonic chemistry.

Everything you just computed is the classical-field limit of that model.

# Part 6: Example 3 -- donor--acceptor electron transfer

## ⚗️ The chemistry

Consider an electron sitting on a donor site D, which can transfer to an acceptor site A:

$$
\mathrm{D}^- \!-\! \mathrm{A} \;\longrightarrow\; \mathrm{D} \!-\! \mathrm{A}^-
$$

Our two states are the **diabatic** (charge-localized) states $|D\rangle \equiv |\alpha\rangle$ and $|A\rangle \equiv |\beta\rangle$.

Here is what is new compared to Example 1. The energies of these two states are not fixed numbers -- they depend on where the **nuclei** are. Squeeze solvent molecules around the negative charge and you stabilize whichever site currently holds the electron. So we introduce a single collective nuclear coordinate $q$ (the "solvent coordinate" or "reaction coordinate"), a linear combination of all the nuclear displacements that matter, and model each diabatic surface as a harmonic well:

$$
V_D(q) = \tfrac{1}{2}k(q+q_0)^2, \qquad V_A(q) = \tfrac{1}{2}k(q-q_0)^2 + \Delta G^{\circ}
$$

The donor state is happiest at $q = -q_0$; the acceptor state at $q = +q_0$; $\Delta G^\circ$ is the driving force. The electronic coupling $V_{DA} = \langle D|\hat{H}|A\rangle$ between the diabats is essentially independent of $q$.

## 🎯 The Pauli decomposition

$$
\hat{H}(q) = \begin{bmatrix} V_D(q) & V_{DA} \\ V_{DA} & V_A(q)\end{bmatrix}
$$

Work out the components:

$$
d_z(q) = V_D(q) - V_A(q) = 2kq_0\,q - \Delta G^{\circ}, \qquad d_x = 2V_{DA}, \qquad d_y = 0
$$

Introducing the **reorganization energy** $\lambda \equiv 2kq_0^2$ (the energy it costs to distort the nuclei from the donor's optimal geometry to the acceptor's, without transferring the electron):

$$
\boxed{\; d_z(q) = \frac{\lambda}{q_0}\,q - \Delta G^{\circ}, \qquad d_x = 2V_{DA} \;}
$$

**This is the structural heart of the example.** The $\hat\sigma_z$ coefficient is *linear in a nuclear coordinate*, and the $\hat\sigma_x$ coefficient is a constant. Compare it to Example 1, where $d_z$ was a fixed electronegativity difference. Here the molecule sweeps its own $d_z$ back and forth as it vibrates -- so the avoided crossing you plotted in Part 3 is not a hypothetical scan, it is something the molecule physically does.

## 📐 What comes out

- **Adiabatic surfaces** (the Born--Oppenheimer potential energy curves the nuclei actually move on):
  $E_\pm(q) = \bar{V}(q) \pm \frac{1}{2}\sqrt{d_z(q)^2 + 4V_{DA}^2}$
- **The crossing point** is where the diabats are degenerate, $d_z(q^\ddagger) = 0$, i.e. $q^\ddagger = \Delta G^\circ q_0/\lambda$.
- **The adiabatic gap there** is $|d_x| = 2V_{DA}$, exactly as in Part 3.
- **The Marcus activation energy**, measured on the diabatic surface from the donor minimum up to the crossing:
  $$\Delta G^{\ddagger} = \frac{(\lambda + \Delta G^{\circ})^2}{4\lambda}$$
  You will verify this numerically below. Note the famous consequence: as the reaction becomes *more* exergonic past $-\Delta G^\circ = \lambda$, the barrier goes back **up**. That is the Marcus **inverted region**, and its experimental confirmation won the 1992 Nobel Prize in Chemistry.

### 🚧 Your Task: build the surfaces

Use $k = 1$, $q_0 = 1$, $\Delta G^\circ = -0.5$, $V_{DA} = 0.15$, all in arbitrary energy units. Write a function returning $\hat{H}(q)$, then plot the two diabats and the two adiabats on the same axes.

In [ ]:
# model parameters (arbitrary energy units)
k_force = 1.0
q0 = 1.0
dG0 = -0.5
V_DA = 0.15
lam = 2 * k_force * q0**2          # reorganization energy

print(f"reorganization energy lambda = {lam}")
print(f"driving force      dG0       = {dG0}")
print(f"electronic coupling V_DA     = {V_DA}")


def V_donor(q):
    """Diabatic potential energy of the charge-on-donor state."""
    # TODO: return 0.5 * k * (q + q0)^2
    ...


def V_acceptor(q):
    """Diabatic potential energy of the charge-on-acceptor state."""
    # TODO: return 0.5 * k * (q - q0)^2 + dG0
    ...


def H_et(q):
    """The 2x2 electron-transfer Hamiltonian at nuclear coordinate q."""
    # TODO: build it. You can use np.array directly, or use build_two_level with
    #       e0 = (V_D + V_A)/2,  dz = V_D - V_A,  dx = 2*V_DA
    ...

In [ ]:
# ✅ Check that d_z(q) really is linear in q, with the slope the theory predicts
for q_test in [-1.7, 0.0, 0.9, 2.3]:
    e0_t, dx_t, dy_t, dz_t = pauli_decomposition(H_et(q_test))
    assert np.isclose(dz_t, (lam / q0) * q_test - dG0), f"d_z wrong at q = {q_test}"
    assert np.isclose(dx_t, 2 * V_DA), "d_x should be 2*V_DA everywhere"
    assert np.isclose(dy_t, 0.0), "d_y should vanish"
print("✅ d_z(q) = (lambda/q0) q - dG0 and d_x = 2 V_DA, confirmed numerically.")

In [ ]:
q_grid = np.linspace(-2.5, 2.5, 601)

# TODO: compute the two adiabatic surfaces at every q
#       hint: la.eigvalsh(H_et(q)) returns both eigenvalues; np.sort keeps them ordered
E_ad = ...                # should have shape (601, 2)
E_lower_ad = ...
E_upper_ad = ...

# TODO: predicted crossing point and Marcus barrier
q_cross = ...
dG_dagger = ...

plt.figure(figsize=(8, 5.4))

# adiabats first, as thick soft bands...
plt.plot(q_grid, E_lower_ad, color="royalblue", lw=5, alpha=0.45, label=r"adiabat $E_-$", zorder=1)
plt.plot(q_grid, E_upper_ad, color="crimson", lw=5, alpha=0.45, label=r"adiabat $E_+$", zorder=1)
# ...then the diabats crisply on top, so you can see where they differ
plt.plot(q_grid, V_donor(q_grid), "k--", lw=1.3, zorder=3,
         label=r"diabat $V_D$  (charge on donor)")
plt.plot(q_grid, V_acceptor(q_grid), color="dimgray", ls=":", lw=1.6, zorder=3,
         label=r"diabat $V_A$  (charge on acceptor)")

# mark the avoided crossing
E_lo_c = np.interp(q_cross, q_grid, E_lower_ad)
E_hi_c = np.interp(q_cross, q_grid, E_upper_ad)
plt.annotate("", xy=(q_cross, E_hi_c), xytext=(q_cross, E_lo_c),
             arrowprops=dict(arrowstyle="<->", color="darkgreen", lw=1.6), zorder=4)
plt.text(q_cross + 0.12, (E_hi_c + E_lo_c) / 2 - 0.03, r"$2V_{DA}$",
         color="darkgreen", fontsize=12, zorder=4)

# mark the Marcus barrier on the donor diabat
plt.annotate("", xy=(-q0 - 0.25, dG_dagger), xytext=(-q0 - 0.25, 0.0),
             arrowprops=dict(arrowstyle="<->", color="darkorange", lw=1.6), zorder=4)
plt.plot([-q0 - 0.32, q_cross], [dG_dagger, dG_dagger], color="darkorange",
         ls="-.", lw=1.0, zorder=2)
plt.plot([-q0 - 0.32, -q0], [0, 0], color="darkorange", ls="-.", lw=1.0, zorder=2)
plt.text(-q0 - 0.34, dG_dagger / 2 - 0.03, r"$\Delta G^{\ddagger}$",
         color="darkorange", fontsize=12, zorder=6, ha="right",
         bbox=dict(facecolor="white", edgecolor="none", pad=1.0))

# mark the two diabatic minima
plt.plot([-q0, q0], [0.0, dG0], "o", color="black", ms=5, zorder=5)
plt.text(-q0, -0.16, r"$\mathrm{D}^-\!\!-\!\mathrm{A}$", ha="center", fontsize=11)
plt.text(q0, dG0 - 0.16, r"$\mathrm{D}\!-\!\mathrm{A}^-$", ha="center", fontsize=11)

plt.xlabel(r"nuclear (solvent) coordinate  $q$")
plt.ylabel("energy")
plt.title("Electron transfer: diabatic vs adiabatic surfaces")
plt.xlim(-2.2, 2.2)
plt.ylim(-0.95, 1.9)
plt.legend(fontsize=9, loc="upper center", ncol=2)
plt.tight_layout()
plt.show()

print(f"crossing at q* = {q_cross:.4f}")
print(f"Marcus barrier (lambda + dG0)^2 / 4 lambda = {dG_dagger:.4f}")
print(f"adiabatic gap at q*                        = {np.interp(q_cross, q_grid, E_upper_ad - E_lower_ad):.4f}"
      f"   (2 V_DA = {2*V_DA})")

In [ ]:
# ✅ Tests on the electron-transfer surfaces

# the minimum adiabatic gap should be exactly 2*V_DA, and occur at q*
gap = E_upper_ad - E_lower_ad
assert np.isclose(np.min(gap), 2 * V_DA, atol=1e-4), "minimum gap should be 2 V_DA"
assert np.isclose(q_grid[np.argmin(gap)], q_cross, atol=5e-3), "gap should be smallest at the crossing"

# the diabats really are degenerate at q*
assert np.isclose(V_donor(q_cross), V_acceptor(q_cross)), "diabats should cross at q*"

# and the Marcus barrier formula is just the diabatic energy at the crossing
assert np.isclose(dG_dagger, V_donor(q_cross) - V_donor(-q0)), \
    "the Marcus barrier IS the donor diabat evaluated at the crossing point"

print("✅ Crossing point, adiabatic gap, and the Marcus barrier formula all verified.")

### 📉 The Marcus inverted region

The barrier formula $\Delta G^\ddagger = (\lambda + \Delta G^\circ)^2/4\lambda$ is a **parabola in the driving force**, minimized at $\Delta G^\circ = -\lambda$. Since the nonadiabatic rate goes as $k_{ET} \propto \exp(-\Delta G^\ddagger/k_BT)$, making a reaction more exergonic speeds it up only until $-\Delta G^\circ = \lambda$, after which it slows down again.

Plot the barrier and the resulting rate against driving force, and locate the inverted region.

In [ ]:
dG_scan = np.linspace(-3 * lam, 0.5 * lam, 400)

# TODO: the Marcus barrier at each driving force
barrier = ...

# TODO: the (unnormalized) nonadiabatic rate, with kT = 0.1 in our energy units
kT = 0.1
rate = ...

fig, (axa, axb) = plt.subplots(1, 2, figsize=(11, 4.2))

axa.plot(dG_scan / lam, barrier / lam, color="darkorange", lw=2)
axa.axvline(-1.0, color="gray", ls=":", lw=1)
axa.set_xlabel(r"$\Delta G^\circ / \lambda$")
axa.set_ylabel(r"$\Delta G^{\ddagger} / \lambda$")
axa.set_title("Activation barrier")
axa.text(-1.9, 0.6, "inverted\nregion", ha="center", fontsize=10, color="gray")
axa.text(-0.35, 0.6, "normal\nregion", ha="center", fontsize=10, color="gray")

axb.semilogy(dG_scan / lam, rate, color="purple", lw=2)
axb.axvline(-1.0, color="gray", ls=":", lw=1)
axb.set_xlabel(r"$\Delta G^\circ / \lambda$")
axb.set_ylabel(r"rate  $\propto e^{-\Delta G^{\ddagger}/k_BT}$")
axb.set_title(r"Marcus rate ($k_BT = 0.1$)")

plt.tight_layout()
plt.show()

print(f"barrier is minimized at dG0/lambda = {dG_scan[np.argmin(barrier)] / lam:.4f}  (theory: -1)")
print(f"minimum barrier                    = {np.min(barrier):.2e}  (theory: 0)")

In [ ]:
# ✅ Tests on the inverted region
assert np.isclose(dG_scan[np.argmin(barrier)], -lam, atol=2e-2), "barrier should be minimized at dG0 = -lambda"
assert np.isclose(np.min(barrier), 0.0, atol=1e-4), "at dG0 = -lambda the barrier vanishes"
# the rate is NOT monotonic in driving force -- that is the whole point
assert np.max(rate) > rate[0] and np.max(rate) > rate[-1], "the rate should peak in the interior"
print("✅ Marcus inverted region reproduced.")

## ⚡ Landau--Zener: sweeping through the crossing

Everything above is static -- surfaces at fixed $q$. Now let the molecule actually *move* through the crossing.

Suppose the nuclear coordinate sweeps linearly in time, $q(t) = vt$, so that

$$
d_z(t) = \alpha t, \qquad \alpha = \frac{\lambda v}{q_0} = \text{the rate of change of the diabatic energy gap}
$$

Start far to one side in a single diabatic state and sweep all the way through. Two things can happen:

- **Adiabatic passage:** the system follows the lower adiabatic surface, and *changes* diabatic character -- the electron transfers.
- **Diabatic passage:** the system blasts straight through the avoided crossing, staying on the same diabat -- no transfer.

Zener's 1932 result gives the probability of the *diabatic* outcome exactly:

$$
\boxed{\;P_{\text{diabatic}} = \exp\!\left(-\frac{2\pi V_{DA}^{2}}{\hbar\,\alpha}\right)\;}
$$

Slow sweep or strong coupling $\Rightarrow$ adiabatic ($P \to 0$). Fast sweep or weak coupling $\Rightarrow$ diabatic ($P \to 1$). The exponent is (coupling)$^2$ over (sweep rate), which is the same combination that appears in Fermi's golden rule and in the nonadiabatic Marcus rate.

### 🚧 Your Task: verify Zener's formula numerically

$\hat{H}(t)$ is now genuinely time dependent, so `propagate` no longer applies directly. The standard fix is to **slice time finely enough that $\hat{H}$ is effectively constant over each slice**, and chain the propagators together:

$$
|\psi(t+\delta t)\rangle \approx \hat{U}\big(\hat{H}(t + \tfrac{\delta t}{2}),\, \delta t\big)\,|\psi(t)\rangle
$$

Evaluating $\hat{H}$ at the **midpoint** of each slice rather than the start makes the error $O(\delta t^3)$ per step instead of $O(\delta t^2)$ -- the same reason the midpoint rule beats the left-endpoint rule for integrals. So we can build the whole thing out of `propagate`, one tiny step at a time.

In [ ]:
def landau_zener_numeric(V_coupling, alpha_sweep, t_max=60.0, n_steps=8000):
    """
    Propagate through a linear avoided crossing and return the probability of
    remaining in the initial DIABATIC state.

    Arguments
    ---------
    V_coupling  : float, the diabatic coupling V_DA (so d_x = 2*V_coupling)
    alpha_sweep : float, the sweep rate, so that d_z(t) = alpha_sweep * t
    t_max       : float, sweep runs from -t_max to +t_max
    n_steps     : int, number of time slices

    Returns
    -------
    P_diabatic : float, |<alpha|psi(+t_max)>|^2
    """
    times = np.linspace(-t_max, t_max, n_steps + 1)
    dt = times[1] - times[0]
    psi = ket_alpha.copy()

    for t in times[:-1]:
        # TODO: evaluate the Hamiltonian at the MIDPOINT of this slice
        t_mid = ...
        H_slice = ...

        # TODO: advance the state by one slice using propagate.
        #       propagate returns shape (1, 2), so reshape back to a 2x1 column.
        psi = ...

    # TODO: the probability of still being in |alpha>
    return ...

In [ ]:
# This cell takes a few seconds to run.
V_list = np.linspace(0.02, 0.50, 10)
alpha_sweep = 1.0

# TODO: numerical and analytic diabatic survival probabilities
P_numeric = ...
P_analytic = ...

plt.figure(figsize=(7, 4.5))
plt.plot(V_list, P_analytic, color="black", lw=2, label=r"Zener: $e^{-2\pi V_{DA}^2/\hbar\alpha}$")
plt.plot(V_list, P_numeric, "o", color="crimson", ms=7, label="numerical propagation")
plt.xlabel(r"diabatic coupling  $V_{DA}$")
plt.ylabel(r"$P_{\rm diabatic}$")
plt.title(r"Landau--Zener transition probability ($\alpha = 1$)")
plt.legend()
plt.tight_layout()
plt.show()

for V, pn, pa in zip(V_list, P_numeric, P_analytic):
    print(f"V_DA = {V:.3f}   numeric = {pn:.4f}   Zener = {pa:.4f}   diff = {abs(pn-pa):.4f}")

In [ ]:
# ✅ Tests for Landau-Zener
assert np.max(np.abs(P_numeric - P_analytic)) < 0.03, \
    "numerics should track the Zener formula to within a few percent"
assert P_numeric[0] > 0.95, "very weak coupling should give near-perfect diabatic passage"
assert P_numeric[-1] < 0.6, "strong coupling should give substantial adiabatic transfer"
print("✅ Landau-Zener verified.")
print()
print("Note the residual few-percent wiggle. Zener's formula is exact only for a sweep")
print("that runs from t = -infinity to +infinity; ours is truncated at finite t_max, so")
print("there is a little leftover interference. Try increasing t_max and n_steps and")
print("watch the agreement improve.")

### 🤔 Questions to consider

**Q17.** In the Landau--Zener formula, $V_{DA}$ appears squared but the sweep rate appears to the first power. Use dimensional analysis on $2\pi V_{DA}^2/\hbar\alpha$ to confirm the exponent is dimensionless, and explain in words why *slower* sweeps favour adiabatic passage.

**Q18.** Marcus theory in the **nonadiabatic** limit has a rate proportional to $|V_{DA}|^2$. Where does that factor come from in the Landau--Zener picture? (Hint: expand $1 - P_{\rm diabatic}$ for small $V_{DA}$.)

**Q19.** For a fixed sweep rate, is there any value of $V_{DA}$ for which the passage is *exactly* adiabatic? What does that imply about the validity of the Born--Oppenheimer approximation near a conical intersection?

**Q20.** Our sweep was linear and one-way. A real vibrating molecule oscillates *back and forth* through the crossing many times. Sketch what you would expect the population to do. (The interference between successive passages is called a Stückelberg oscillation, and it is a real, measured effect.)

### 🔮 Looking ahead: this is the Holstein model waiting to happen

Look once more at the boxed result for this example:

$$
\hat{H}(q) = \bar{V}(q)\,\hat{1} + \frac{1}{2}\left[\frac{\lambda}{q_0}q - \Delta G^\circ\right]\hat\sigma_z + V_{DA}\,\hat\sigma_x
$$

We treated $q$ as a **classical parameter** -- a number we dial, or sweep linearly in time. But $q$ is a nuclear coordinate, and nuclei are quantum. Promote it to a harmonic oscillator, $\hat{q} \propto (\hat{a} + \hat{a}^\dagger)$, and the Hamiltonian becomes

$$
\hat{H}_{\rm Holstein} = \frac{\varepsilon}{2}\hat\sigma_z + V_{DA}\,\hat\sigma_x + \hbar\omega\,\hat{a}^\dagger\hat{a}
+ \hbar\omega\, g\,(\hat{a}+\hat{a}^\dagger)\,\hat\sigma_z
$$

the **Holstein** (or spin--boson) Hamiltonian. The term that used to be "the nuclear coordinate tunes the site energies" becomes a genuine coupling between the electronic two-level system and a quantized vibration.

Notice the contrast with the Jaynes--Cummings model at the end of Part 5:

| | coupling operator | physical origin |
|---|---|---|
| Jaynes--Cummings | $\hat{a}\hat\sigma_+ + \hat{a}^\dagger\hat\sigma_-$ | field **flips** the two-level system |
| Holstein | $(\hat{a}+\hat{a}^\dagger)\hat\sigma_z$ | vibration **shifts** the two-level system's energies |

Same two-dimensional electronic space, same boson, completely different physics -- one gives polaritons and Rabi splitting, the other gives vibronic progressions, Franck--Condon factors, and polaron formation. You have now built the single-spin ancestor of both.

# Part 7: A fourth example -- ammonia inversion

## 🌂 Tunneling as a two-level problem

Everything so far has used a two-level space built from two *different places an electron can be*. Here is one built from two *different shapes the same molecule can have*.

Ammonia is pyramidal, and the nitrogen can pop through the plane of the three hydrogens like an umbrella turning inside out. Call the two pyramidal geometries $|L\rangle$ ("umbrella down") and $|R\rangle$ ("umbrella up"). Classically they are separated by a barrier of about 2000 cm$^{-1}$ and the molecule would be stuck in one of them. Quantum mechanically the nitrogen **tunnels**, and that tunneling is precisely an off-diagonal matrix element:

$$
\hat{H} = \begin{bmatrix} 0 & -\Delta_t \\ -\Delta_t & 0 \end{bmatrix} = -\Delta_t\,\hat\sigma_x
$$

Pure $\hat\sigma_x$. The eigenstates are the symmetric and antisymmetric combinations $\frac{1}{\sqrt2}(|L\rangle \pm |R\rangle)$, split by $2\Delta_t$. This is the famous **inversion doublet** of ammonia, and it is measured to be

$$
2\Delta_t = 0.7934~\mathrm{cm}^{-1} = 23.786~\mathrm{GHz}
$$

which is a microwave transition -- the line that Townes and coworkers used to build the first **maser** in 1954, three years before the laser.

Notice the crucial point: the physically stable, energy-eigenstate configurations of ammonia are *not* the pyramidal geometries. They are delocalized over both. A molecule prepared as $|L\rangle$ is a superposition of the two eigenstates, so from Part 4 it oscillates -- the umbrella flips back and forth at 23.786 GHz.

## ⚡ Adding a $\hat\sigma_z$ term: the Stark effect

Now switch on a static electric field $\mathcal{E}$ along the molecular axis. The two pyramidal geometries have their dipole moments pointing in **opposite** directions, so the field stabilizes one and destabilizes the other:

$$
\hat{H} = \mu\mathcal{E}\,\hat\sigma_z - \Delta_t\,\hat\sigma_x
\qquad\Longrightarrow\qquad d_z = 2\mu\mathcal{E}, \quad d_x = -2\Delta_t
$$

with $\mu = 1.47$ D the NH$_3$ dipole moment. This is exactly the structure of Example 1, but now **you** control $d_z$ with a knob on the bench. Turn the field up and you drive the molecule from delocalized ($\theta = \pi/2$) to localized ($\theta \to 0$): the field literally traps the umbrella on one side.

The energy gap is $\sqrt{(2\mu\mathcal{E})^2 + (2\Delta_t)^2}$ -- quadratic in $\mathcal{E}$ at low field (a second-order Stark effect), crossing over to linear at high field, and the crossover happens at the field where $\mu\mathcal{E} = \Delta_t$.

### 🚧 Your Task

Work in cm$^{-1}$ and express the field in kV/cm. The conversion you need is given.

Compute the crossover field, then plot the two levels and the degree of localization against field strength.

In [ ]:
# NH3 parameters
inversion_splitting = 0.7934        # cm^-1, the measured 2*Delta_t
Delta_t = inversion_splitting / 2   # cm^-1

# dipole conversion:  mu = 1.47 D expressed in cm^-1 per (kV/cm)
#   1 D = 3.33564e-30 C m ;  E = 1 kV/cm = 1e5 V/m ;  hc = 1.98645e-23 J cm
mu_cm1_per_kVcm = 1.47 * 3.33564e-30 * 1e5 / 1.98645e-23

print(f"tunneling matrix element Delta_t = {Delta_t:.5f} cm^-1")
print(f"inversion splitting 2*Delta_t    = {inversion_splitting:.4f} cm^-1"
      f" = {inversion_splitting * 29.9792458:.3f} GHz")
print(f"dipole coupling                  = {mu_cm1_per_kVcm:.5f} cm^-1 per (kV/cm)")

# TODO: the crossover field, where mu*E equals Delta_t
E_crossover = ...
print(f"crossover field                  = {E_crossover:.2f} kV/cm")


def H_ammonia(E_field_kVcm):
    """Two-level Hamiltonian for NH3 inversion in a static field, in cm^-1."""
    # TODO: d_z = 2*mu*E and d_x = -2*Delta_t
    ...

In [ ]:
E_scan = np.linspace(0, 60, 400)          # kV/cm

# TODO: at each field, get the two eigenvalues and the localization of the LOWER state.
#       "Localization" here is |<sigma_z>| of the lower eigenvector: 0 means fully
#       delocalized over both umbrella geometries, 1 means fully trapped on one side.
#       hint: la.eigh(H)[1][:, [0]] is the lower eigenvector as a 2x1 column
E_levels = ...                # shape (400, 2)
localization = ...            # shape (400,)

fig, (axL, axR) = plt.subplots(1, 2, figsize=(11, 4.2))

axL.plot(E_scan, E_levels[:, 1], color="crimson", lw=2, label="upper level")
axL.plot(E_scan, E_levels[:, 0], color="royalblue", lw=2, label="lower level")
axL.plot(E_scan, mu_cm1_per_kVcm * E_scan, "k--", lw=1, label=r"$\pm\mu\mathcal{E}$ (no tunneling)")
axL.plot(E_scan, -mu_cm1_per_kVcm * E_scan, "k--", lw=1)
axL.axvline(E_crossover, color="gray", ls=":", lw=1)
axL.set_xlabel(r"electric field  $\mathcal{E}$  (kV/cm)")
axL.set_ylabel(r"energy (cm$^{-1}$)")
axL.set_title("Stark effect on the NH$_3$ inversion doublet")
axL.legend(fontsize=9)

axR.plot(E_scan, localization, color="darkgreen", lw=2)
axR.axvline(E_crossover, color="gray", ls=":", lw=1)
axR.text(E_crossover + 1.5, 0.15, r"$\mu\mathcal{E} = \Delta_t$", color="gray", fontsize=10)
axR.set_xlabel(r"electric field  $\mathcal{E}$  (kV/cm)")
axR.set_ylabel(r"$|\langle\sigma_z\rangle|$  of the lower state")
axR.set_title("The field traps the umbrella on one side")
axR.set_ylim(-0.03, 1.03)

plt.tight_layout()
plt.show()

print(f"gap at zero field       = {E_levels[0, 1] - E_levels[0, 0]:.4f} cm^-1"
      f"   (should be {inversion_splitting})")
gap_cross = np.interp(E_crossover, E_scan, E_levels[:, 1] - E_levels[:, 0])
print(f"gap at crossover field  = {gap_cross:.4f} cm^-1"
      f"   (should be sqrt(2) * {inversion_splitting} = {np.sqrt(2)*inversion_splitting:.4f})")

In [ ]:
# ✅ Tests for the ammonia Stark effect
assert np.isclose(inversion_splitting * 29.9792458, 23.786, atol=0.02), \
    "0.7934 cm^-1 should be the 23.786 GHz maser line"
assert np.isclose(E_levels[0, 1] - E_levels[0, 0], inversion_splitting), \
    "zero-field gap must be the inversion splitting"
assert np.isclose(gap_cross, np.sqrt(2) * inversion_splitting, atol=1e-3), \
    "at the crossover field the gap should grow by exactly sqrt(2)"
assert localization[0] < 1e-8, "at zero field the eigenstates are fully delocalized"
assert localization[-1] > 0.9, "at high field the lower state should be strongly localized"
print("✅ NH3 inversion doublet and its Stark tuning reproduced.")

### 🤔 Questions to consider

**Q21.** At zero field the ground state is $\frac{1}{\sqrt2}(|L\rangle+|R\rangle)$, which has **zero dipole moment** even though NH$_3$ is famously a polar molecule with $\mu = 1.47$ D. Resolve this apparent contradiction. (What is actually measured in a dipole-moment experiment, and on what timescale?)

**Q22.** The crossover field you computed is around 16 kV/cm. Deuterating to ND$_3$ drops the inversion splitting to about 0.053 cm$^{-1}$. What happens to the crossover field, and why is the splitting so sensitive to isotopic substitution?

**Q23.** Compare the three "$\sigma_x$" quantities you have now met: the resonance integral $\beta$, the electronic coupling $V_{DA}$, and the tunneling element $\Delta_t$. In what sense are they the same quantity? In what sense are they different?

# Part 8: Where does $\hat\sigma_y$ actually come from?

You may have noticed something. Of our four examples, **only the driven spin had a $\hat\sigma_y$ term.** The LCAO dimer, the electron-transfer complex, and ammonia all had $d_y = 0$. That is not a coincidence, and the reason is worth understanding.

## 🪞 The gauge argument: $\hat\sigma_y$ alone is not physical

Here is an uncomfortable fact. Take any Hamiltonian with $d_x$ and $d_y$ both nonzero, and rephase your second basis state:

$$
|\beta\rangle \;\longrightarrow\; e^{i\chi}|\beta\rangle
$$

This is allowed -- the overall phase of a basis vector is a convention, not physics. Under this rephasing the off-diagonal element picks up $e^{i\chi}$, which rotates $(d_x, d_y)$ in the plane. Choose $\chi = \varphi = \mathrm{atan2}(d_y, d_x)$ and you can always make $d_y$ **exactly zero**.

So for a *single, static* Hamiltonian, a $\hat\sigma_y$ term carries no physical content whatsoever. You will verify this numerically below. Everything real depends only on $|\mathbf{d}| = \sqrt{d_z^2 + d_x^2 + d_y^2}$, which the rephasing leaves alone.

## 🕰️ When it *does* matter

$\hat\sigma_y$ acquires meaning only **relative to something else**, when there is a second thing in the problem that fixes the phase convention:

1. **A time-dependent drive.** In Part 5 the drive phase $\varphi$ moved $\mathbf{d}$ around the $xy$ plane. Any *single* pulse can be called an "$x$-pulse" by convention -- but once you have fixed that, the phase of the *second* pulse in a sequence is physical. This is exactly why two-dimensional NMR works.
2. **Broken time-reversal symmetry.** For a real electronic Hamiltonian in a basis of real orbitals with no magnetic field, time-reversal symmetry forces $\hat{H}$ to be **real symmetric**, so $d_y = 0$ automatically. To get a genuine, unremovable $\hat\sigma_y$ you need something that breaks it: a magnetic field (a Peierls phase on the hopping integral), circularly polarized light, or **spin--orbit coupling**, which is why the effective $2\times2$ Hamiltonian of a Kramers doublet in a heavy-atom complex generally has all three Pauli components.
3. **Three or more coupled states.** In a triangle of coupled sites you can rephase away at most two of the three complex hoppings. The leftover phase around the loop is gauge invariant -- it is a magnetic flux, and it is the origin of the Aharonov--Bohm effect and of Berry phase.

The takeaway to carry into the rest of the course: **when you meet a real electronic-structure Hamiltonian in a real basis, expect $\hat\sigma_z$ and $\hat\sigma_x$ only.** A $\hat\sigma_y$ term is a signal that something interesting is going on.

### 🚧 Your Task: rephase $\hat\sigma_y$ away

Take a Hamiltonian with all three components nonzero. Build the diagonal phase matrix $\hat{U} = \mathrm{diag}(1, e^{i\chi})$, transform via $\hat{H}' = \hat{U}^\dagger \hat{H}\hat{U}$, and choose $\chi$ to kill $d_y$. Then confirm the eigenvalues are unchanged.

In [ ]:
# a Hamiltonian with all three Pauli components
e0_g, dz_g, dx_g, dy_g = 0.2, 1.0, 0.6, -0.8
H_gauge = build_two_level(e0_g, dz_g, dx_g, dy_g)

# TODO: the phase that rotates (dx, dy) onto the +x axis
chi = ...

# TODO: the diagonal rephasing matrix U = diag(1, exp(i*chi))
U_phase = ...

# TODO: the transformed Hamiltonian H' = U^dagger H U
H_gauged = ...

e0_p, dx_p, dy_p, dz_p = pauli_decomposition(H_gauged)

print(f"before:  dx = {dx_g:+.4f}   dy = {dy_g:+.4f}   dz = {dz_g:+.4f}   |d_perp| = {np.hypot(dx_g, dy_g):.4f}")
print(f"after:   dx = {dx_p:+.4f}   dy = {dy_p:+.4f}   dz = {dz_p:+.4f}")
print()
print(f"eigenvalues before: {np.sort(la.eigvalsh(H_gauge))}")
print(f"eigenvalues after:  {np.sort(la.eigvalsh(H_gauged))}")

In [ ]:
# ✅ Tests for the gauge argument
assert abs(dy_p) < 1e-12, "the rephasing should have eliminated d_y entirely"
assert np.isclose(dx_p, np.hypot(dx_g, dy_g)), "d_x should absorb the whole transverse magnitude"
assert np.isclose(dz_p, dz_g), "a diagonal rephasing cannot touch d_z"
assert np.allclose(np.sort(la.eigvalsh(H_gauge)), np.sort(la.eigvalsh(H_gauged))), \
    "a unitary transformation cannot change the spectrum"
assert np.allclose(H_gauged, H_gauged.conj()), "H' should now be purely real"
print("✅ For a single static Hamiltonian, sigma_y is pure convention.")

### 🤔 Questions to consider

**Q24.** The rephasing matrix $\hat{U} = \mathrm{diag}(1, e^{i\chi})$ can also be written as $e^{i\chi/2}e^{-i\chi\hat\sigma_z/2}$. Compare that to the rotating-frame transformation in Part 5. What is the physical difference between a *constant* $\chi$ and a *time-dependent* $\chi = \omega t$?

**Q25.** We showed $\hat{H}'$ came out purely real. Argue that any $2\times2$ Hermitian matrix can be made real symmetric this way, and then explain why the same trick does **not** work for a general $3\times3$ Hermitian matrix. (Count the phases you can adjust against the phases you need to remove.)

# Part 9: Putting it together

## 📋 Summary: one Hamiltonian, four molecules

Every example in this notebook is the same equation, $\hat{H} = \varepsilon_0\hat{1} + \frac{1}{2}\mathbf{d}\cdot\hat{\boldsymbol\sigma}$. Only the dictionary changes.

| System | $|\alpha\rangle,\ |\beta\rangle$ | $d_z$ (asymmetry) | $d_x, d_y$ (coupling) | Splitting $|\mathbf{d}|$ is called... |
|---|---|---|---|---|
| **LCAO dimer** | AO on atom A / atom B | $\alpha_A - \alpha_B$ (electronegativity) | $2\beta$ (resonance integral) | bonding--antibonding gap |
| **Driven spin** | spin up / spin down | $\hbar(\omega_0-\omega)$ (detuning) | $\hbar\omega_1 e^{\pm i\varphi}$ (drive) | generalized Rabi frequency $\hbar\Omega$ |
| **Electron transfer** | charge on D / on A | $(\lambda/q_0)q - \Delta G^\circ$ (nuclear coordinate!) | $2V_{DA}$ (electronic coupling) | adiabatic gap |
| **NH$_3$ inversion** | umbrella down / up | $2\mu\mathcal{E}$ (Stark field) | $-2\Delta_t$ (tunneling) | inversion doublet |

And the physics you get is always the same three statements:

1. **Splitting.** $E_\pm = \varepsilon_0 \pm \frac{1}{2}|\mathbf{d}|$. Coupled levels never cross; the closest they get is $|d_\perp|$.
2. **Mixing.** $\tan\theta = d_\perp/d_z$ decides whether the eigenstates are localized on the basis states or delocalized between them.
3. **Dynamics.** $\dot{\mathbf{s}} = \frac{1}{\hbar}\mathbf{d}\times\mathbf{s}$. The state precesses about $\mathbf{d}$ at the Bohr frequency.

Whenever you meet a new two-level problem -- an exciton dimer, a qubit, a Kramers doublet, a curve crossing in a photochemical reaction -- the first thing to do is find $\mathbf{d}$.

## 🔮 What comes next

Both of the models we teased live in a **bigger Hilbert space**: spin $\otimes$ boson, not spin alone. That is the essential step you have not taken yet.

| Model | Coupling term | You met its ancestor in... |
|---|---|---|
| **Jaynes--Cummings** | $\hbar g(\hat{a}\hat\sigma_+ + \hat{a}^\dagger\hat\sigma_-)$ | Part 5, with the field as a classical number |
| **Holstein / spin--boson** | $\hbar\omega g(\hat{a}+\hat{a}^\dagger)\hat\sigma_z$ | Part 6, with $q$ as a classical parameter |

In both cases the two-level machinery you built here survives intact -- it just gets tensored with a harmonic oscillator. Keep `build_two_level`, `pauli_decomposition`, `analytic_eigensystem`, `bloch_vector`, and `propagate`; you will use them again.

## 📝 Practice Problems

**P1. The excitonic dimer.**
Two identical chromophores sit close enough that their transition dipoles interact. In the basis $\{|e_1 g_2\rangle, |g_1 e_2\rangle\}$ (molecule 1 excited / molecule 2 excited) the singly-excited Hamiltonian is

$$
\hat{H} = \begin{bmatrix} \varepsilon_1 & J \\ J & \varepsilon_2 \end{bmatrix}
$$

where $J$ is the excitonic coupling.

(a) Identify $\varepsilon_0$, $d_z$, and $d_x$.

(b) For identical chromophores ($\varepsilon_1 = \varepsilon_2$), find the eigenstates and their energies. The splitting $2|J|$ is called the **Davydov splitting**.

(c) The transition dipole to the symmetric state is $\boldsymbol\mu_1 + \boldsymbol\mu_2$ and to the antisymmetric state is $\boldsymbol\mu_1 - \boldsymbol\mu_2$. For a "head-to-tail" (J-aggregate) geometry $J < 0$ and for a "side-by-side" (H-aggregate) geometry $J > 0$. In each case, is the *bright* state the upper one or the lower one? Predict whether the absorption spectrum red-shifts or blue-shifts relative to the monomer.

(d) Modify the code from Part 3 to compute and plot the two exciton energies and their oscillator strengths as a function of $\varepsilon_1 - \varepsilon_2$ for fixed $J$.

---

**P2. Hückel butadiene is two dimers.**
The $\pi$ system of *trans*-butadiene in Hückel theory is a $4\times4$ problem, but it has a symmetry: reflection through the midpoint of the central bond.

(a) Form the symmetric and antisymmetric combinations $\frac{1}{\sqrt2}(p_1 \pm p_4)$ and $\frac{1}{\sqrt2}(p_2 \pm p_3)$, and show that in this basis the $4\times4$ Hamiltonian **block-diagonalizes** into two $2\times2$ blocks.

(b) Identify $d_z$ and $d_x$ for each block and use `analytic_eigensystem` to get all four $\pi$ orbital energies. Check against the textbook result $E = \alpha \pm 1.618\beta,\ \alpha \pm 0.618\beta$.

(c) Verify numerically by diagonalizing the full $4\times4$ matrix.

*(This "use the symmetry to reduce to two-level blocks" move is one of the most useful tricks in all of quantum chemistry.)*

---

**P3. Stückelberg oscillations.**
Modify `landau_zener_numeric` so that instead of sweeping once, $d_z(t) = A\sin(\omega_{\rm sw} t)$ oscillates back and forth through the crossing many times.

(a) Plot the population in $|\alpha\rangle$ against time for a few periods.

(b) Explain why the population does not simply decay toward 1/2.

(c) Find a sweep amplitude and frequency for which the transfer after two passages is *smaller* than after one. What is interfering with what?

---

**P4. A genuine $\hat\sigma_y$.**
Consider a two-site model in a magnetic field, where the hopping integral acquires a Peierls phase: $\beta \to \beta e^{i\phi_B}$.

(a) Write down $\hat{H}$ and identify $d_x$ and $d_y$.

(b) Show that the eigenvalues do not depend on $\phi_B$, consistent with Part 8.

(c) Now do the same for a *triangle* of three sites, each hopping carrying phase $\phi_B$. Show that the eigenvalues **do** depend on the total flux $3\phi_B$, and that this dependence cannot be rephased away. Diagonalize numerically and plot the three levels against flux.

---

**P5. Two-pulse interferometry (a Ramsey sequence).**
Using the machinery of Part 5, simulate the following on a system with detuning $\Delta$:

1. a $\pi/2$ pulse with phase $\varphi = 0$,
2. free evolution (drive off, so $\mathbf{d} = \hbar\Delta\hat{z}$) for a time $\tau$,
3. a second $\pi/2$ pulse, also with $\varphi = 0$.

(a) Plot the final $P_\beta$ against $\tau$. What is the oscillation frequency?

(b) Repeat with the second pulse phase-shifted to $\varphi = \pi$. What happens, and why does this show that the *relative* phase of two pulses is physical even though the phase of one is not?

(c) This is the Ramsey separated-oscillatory-fields method, and it is how atomic clocks work. Explain in one or two sentences why making $\tau$ longer makes the frequency measurement more precise.

## 📚 References

1. Hückel, E. Quantentheoretische Beiträge zum Benzolproblem. *Z. Phys.* **1931**, *70*, 204--286.

2. Rabi, I. I. Space Quantization in a Gyrating Magnetic Field. *Phys. Rev.* **1937**, *51*, 652--654.

3. Feynman, R. P.; Vernon, F. L.; Hellwarth, R. W. Geometrical Representation of the Schrödinger Equation for Solving Maser Problems. *J. Appl. Phys.* **1957**, *28*, 49--52. *(The paper that introduced the Bloch-vector picture for a general two-level system, not just spins.)*

4. Marcus, R. A. On the Theory of Oxidation-Reduction Reactions Involving Electron Transfer. *J. Chem. Phys.* **1956**, *24*, 966--978.

5. Marcus, R. A. Electron Transfer Reactions in Chemistry: Theory and Experiment (Nobel Lecture). *Rev. Mod. Phys.* **1993**, *65*, 599--610.

6. Zener, C. Non-Adiabatic Crossing of Energy Levels. *Proc. R. Soc. London A* **1932**, *137*, 696--702.

7. Landau, L. D. Zur Theorie der Energieübertragung II. *Phys. Z. Sowjetunion* **1932**, *2*, 46--51.

8. Gordon, J. P.; Zeiger, H. J.; Townes, C. H. The Maser -- New Type of Microwave Amplifier, Frequency Standard, and Spectrometer. *Phys. Rev.* **1955**, *99*, 1264--1274.

9. Robin, M. B.; Day, P. Mixed Valence Chemistry -- A Survey and Classification. *Adv. Inorg. Chem. Radiochem.* **1968**, *10*, 247--422.

10. Creutz, C.; Taube, H. Direct Approach to Measuring the Franck--Condon Barrier to Electron Transfer between Metal Ions. *J. Am. Chem. Soc.* **1969**, *91*, 3988--3989.

11. Jaynes, E. T.; Cummings, F. W. Comparison of Quantum and Semiclassical Radiation Theories with Application to the Beam Maser. *Proc. IEEE* **1963**, *51*, 89--109.

12. Holstein, T. Studies of Polaron Motion: Part I. The Molecular-Crystal Model. *Ann. Phys.* **1959**, *8*, 325--342.

13. Leggett, A. J.; Chakravarty, S.; Dorsey, A. T.; Fisher, M. P. A.; Garg, A.; Zwerger, W. Dynamics of the Dissipative Two-State System. *Rev. Mod. Phys.* **1987**, *59*, 1--85.
